### Helper functions:

In [ ]:
# !pip install requests, json

In [ ]:

import requests
import json
from pprint import pprint
from IPython.display import display, HTML

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "fa-IR,fa;q=0.9,en;q=0.8"
})


def inspect_endpoint(name, url, method="GET", params=None,
                     payload=None, headers=None):

    print("\n" + "=" * 80)
    print(f"PLATFORM: {name}")
    print(f"METHOD: {method}")
    print(f"URL: {url}")
    print("=" * 80)

    try:
        response = session.request(
            method=method,
            url=url,
            params=params,
            json=payload,
            headers=headers,
            timeout=30
        )

        print("\nHTTP STATUS:", response.status_code)
        print("CONTENT TYPE:", response.headers.get("Content-Type"))
        print("RESPONSE SIZE:", len(response.content), "bytes")
        print("FINAL URL:", response.url)

        print("\nRESPONSE BODY")
        print("-" * 80)

        if not response.content:
            print("[Empty response body]")
        else:
            try:
                data = response.json()

                print("VALID JSON RESPONSE")
                print("\nComplete JSON payload:\n")

                print(json.dumps(
                    data,
                    ensure_ascii=False,
                    indent=4
                ))

            except ValueError:
                print("Response is not valid JSON.")
                print("\nRaw response:\n")
                print(response.text)

        if not response.ok:
            print("\nWARNING: HTTP request returned an unsuccessful status.")

        return response

    except requests.RequestException as e:
        print("\nREQUEST ERROR:", repr(e))
        return None


### Bama\.ir

In [ ]:

url = "https://bama.ir/cad/api/search"

for page in range(3):
    inspect_endpoint(
        name=f"Bama.ir - Page {page}",
        url=url,
        params={
            "pageIndex": page,
            "pageSize": 12
        }
    )


### Divar\.ir

In [ ]:

url = "https://api.divar.ir/v8/postlist/w/search"

# GET request
inspect_endpoint(
    name="Divar - Search API (GET)",
    url=url,
    method="GET"
)

# POST request without a body (initial diagnostic only)
inspect_endpoint(
    name="Divar - Search API (POST)",
    url=url,
    method="POST"
)


### Sheypoor

In [ ]:

urls = [
    "https://www.sheypoor.com/api/v10.0.0/search/iran/car?f=%5B1.0%2C%201793807126545%5D_465914613n_1793795579.366213&p=4",

    "https://www.sheypoor.com/api/v10.0.0/search/iran/car?f=%5B1.0%2C%201793806551858%5D_468050469n_1793795579.366213&p=6"
]

for i, url in enumerate(urls, 1):
    inspect_endpoint(
        name=f"Sheypoor - Search API #{i}",
        url=url
    )


### Hamrah Mechanic

In [ ]:

url = "https://www.hamrah-mechanic.com/api/v1/common/newexhibition"

for page in [2, 3, 5, 13, 19, 26]:
    inspect_endpoint(
        name=f"Hamrah Mechanic - Page {page}",
        url=url,
        params={
            "page": page,
            "size": 24
        }
    )


### Karnameh

In [ ]:

listing_url = (
    "https://api-gw.karnameh.com/post-storage/car-posts/"
    "car-post-list/?&size=20&start=0&sort=newest"
    "&relevant=false&is_guaranteed=false"
    "&next_set=1793682405657&next_set=1"
    "&user_id=<karnameh_user_id>"
)

detail_urls = [
    "https://karnameh.com/_next/data/nQ1KnoV5mUaQDMuVRBHOO/used-cars/430de1f3-a59d-44a5-8bc3-c74beb77a4d8.json?id=430de1f3-a59d-44a5-8bc3-c74beb77a4d8",

    "https://karnameh.com/_next/data/nQ1KnoV5mUaQDMuVRBHOO/used-cars/6e140168-2d1b-4b5a-9811-87ba176d2b77.json?id=6e140168-2d1b-4b5a-9811-87ba176d2b77",

    "https://karnameh.com/_next/data/nQ1KnoV5mUaQDMuVRBHOO/used-cars/4385b37e-6ce5-413b-9477-6386b20902bd.json?id=4385b37e-6ce5-413b-9477-6386b20902bd"
]

inspect_endpoint(
    name="Karnameh - Advertisement Listing API",
    url=listing_url
)

for i, url in enumerate(detail_urls, 1):
    inspect_endpoint(
        name=f"Karnameh - Advertisement Detail #{i}",
        url=url
    )


# Fetch, Extract features, from car ads from sources

## Bama\.ir — experimental fetch and extraction

In [ ]:
import pandas as pd
if "requests" not in globals():
    import requests
if "json" not in globals():
    import json
if "session" not in globals():
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "fa-IR,fa;q=0.9,en;q=0.8",
    })

# Bama.ir: fetch one page and retain up to three actual car advertisements.
bama_summary = {"platform": "Bama.ir", "ads_extracted": 0, "p0_populated": [], "p0_unavailable": [], "p1_populated": [], "p2_found": [], "needs_investigation": True, "issue": None}
bama_raw_payloads = {}
bama_rows = []

try:
    bama_url = "https://bama.ir/cad/api/search"
    bama_response = session.get(bama_url, params={"pageIndex": 0, "pageSize": 12}, timeout=30)
    print("Bama HTTP status:", bama_response.status_code)
    bama_response.raise_for_status()
    bama_json = bama_response.json()

    def _bama_walk(node):
        if isinstance(node, dict):
            yield node
            for value in node.values():
                yield from _bama_walk(value)
        elif isinstance(node, list):
            for value in node:
                yield from _bama_walk(value)

    bama_ads = []
    bama_seen = set()
    for node in _bama_walk(bama_json):
        if node.get("type") == "ad" and isinstance(node.get("detail"), dict):
            detail = node["detail"]
            # The captured search payload marks vehicle listings as detail.type == "car".
            if detail.get("type") not in (None, "car"):
                continue
            listing_id = detail.get("code")
            unique_key = listing_id or detail.get("url") or id(node)
            if unique_key not in bama_seen:
                bama_ads.append(node)
                bama_seen.add(unique_key)
            if len(bama_ads) == 3:
                break

    def _bama_nonempty(value):
        return value is not None and value != ""

    for ad in bama_ads:
        detail = ad.get("detail") or {}
        metadata = ad.get("metadata") or {}
        price_obj = ad.get("price") or detail.get("price")
        dealer = ad.get("dealer")
        specs = ad.get("specs")
        images = ad.get("images") or []
        canonical = metadata.get("canonical")
        relative_url = detail.get("url")
        public_url = canonical if isinstance(canonical, str) and canonical.startswith("https://") else None
        if public_url is None and isinstance(relative_url, str) and relative_url.startswith("/"):
            public_url = "https://bama.ir" + relative_url
        elif public_url is None and isinstance(relative_url, str) and relative_url.startswith("https://"):
            public_url = relative_url

        image_urls = []
        if _bama_nonempty(detail.get("image")):
            image_urls.append(detail["image"])
        for image in images if isinstance(images, list) else []:
            if isinstance(image, dict):
                image_urls.extend(value for value in image.values() if isinstance(value, str) and value.startswith("http"))
            elif isinstance(image, str) and image.startswith("http"):
                image_urls.append(image)
        image_urls = list(dict.fromkeys(image_urls))

        row = {
            "source_platform": "Bama.ir",
            "source_listing_id": detail.get("code"),
            "original_url": public_url,
            "url_raw": canonical or relative_url,
            "title": detail.get("title"),
            "description": detail.get("description"),
            "brand_raw": detail.get("brand_fa") if _bama_nonempty(detail.get("brand_fa")) else detail.get("brand"),
            "brand_en_raw": detail.get("brand"),
            "model_raw": detail.get("subtitle"),
            "trim_raw": detail.get("trim"),
            "year_raw": detail.get("year"),
            "mileage_raw": detail.get("mileage"),
            "price_raw": price_obj.get("price") if isinstance(price_obj, dict) else price_obj,
            "price_details": json.dumps(price_obj, ensure_ascii=False) if price_obj is not None else None,
            "location_raw": detail.get("location"),
            "fuel_raw": detail.get("fuel"),
            "transmission_raw": detail.get("transmission"),
            "body_type_raw": detail.get("body_type_fa") if _bama_nonempty(detail.get("body_type_fa")) else detail.get("body_type"),
            "body_condition_raw": detail.get("body_status"),
            "color_raw": detail.get("color"),
            "seller_type": dealer.get("type") if isinstance(dealer, dict) else None,
            "seller_id": dealer.get("id") if isinstance(dealer, dict) else None,
            "seller_name_raw": dealer.get("name") if isinstance(dealer, dict) else None,
            "image_urls": image_urls or None,
            "image_count": detail.get("image_count"),
            "source_published_at": detail.get("time"),
            "source_modified_at": detail.get("modified_date"),
            "specs_raw": json.dumps(specs, ensure_ascii=False) if specs is not None else None,
            "fuel_consumption_raw": specs.get("fuel") if isinstance(specs, dict) else None,
            "dealer_details": json.dumps(dealer, ensure_ascii=False) if dealer is not None else None,
            "detail_extra_raw": json.dumps(detail, ensure_ascii=False),
            "metadata_raw": json.dumps(metadata, ensure_ascii=False) if metadata else None,
        }
        bama_rows.append(row)
        if row["source_listing_id"] is not None:
            bama_raw_payloads[row["source_listing_id"]] = ad

    bama_summary["ads_extracted"] = len(bama_rows)
    if not bama_rows:
        bama_summary["issue"] = "No type='ad' vehicle records with a detail object were found on page 0."

except Exception as exc:
    bama_summary["issue"] = f"Fetch or parse failed: {type(exc).__name__}: {exc}"
    print("Bama experiment issue:", bama_summary["issue"])

bama_df = pd.DataFrame(bama_rows)
bama_p0_fields = ["source_listing_id", "original_url", "title", "description", "brand_raw", "model_raw", "year_raw", "mileage_raw", "price_raw", "location_raw"]
bama_p1_fields = ["trim_raw", "fuel_raw", "transmission_raw", "body_type_raw", "body_condition_raw", "color_raw", "seller_type", "seller_id", "image_urls", "image_count", "source_published_at", "source_modified_at"]
bama_summary["p0_populated"] = [f for f in bama_p0_fields if any(_bama_nonempty(r.get(f)) for r in bama_rows)] if "_bama_nonempty" in globals() else []
bama_summary["p0_unavailable"] = [f for f in bama_p0_fields if f not in bama_summary["p0_populated"]]
bama_summary["p1_populated"] = [f for f in bama_p1_fields if any(_bama_nonempty(r.get(f)) for r in bama_rows)] if "_bama_nonempty" in globals() else []
bama_summary["p2_found"] = [f for f in ["specs_raw", "fuel_consumption_raw", "dealer_details", "metadata_raw"] if any(_bama_nonempty(r.get(f)) for r in bama_rows)] if "_bama_nonempty" in globals() else []
bama_summary["needs_investigation"] = not bool(bama_rows)

print("Bama extraction report")
print("Ads extracted:", bama_summary["ads_extracted"])
print("P0 populated:", bama_summary["p0_populated"])
print("P0 unavailable:", bama_summary["p0_unavailable"])
print("P1 populated:", bama_summary["p1_populated"])
print("P2/source-specific fields found:", bama_summary["p2_found"])
print("Public URL verified from canonical or observed relative listing path:", sum(r.get("original_url") is not None for r in bama_rows), "of", len(bama_rows))
print("Issue:", bama_summary["issue"])
if bama_rows:
    example_id = bama_rows[0].get("source_listing_id")
    print("Example complete raw advertisement JSON for", example_id, ":")
    print(json.dumps(bama_raw_payloads.get(example_id), ensure_ascii=False, indent=2))
bama_df

## Divar\.ir — experimental fetch and diagnostic

In [ ]:
# Divar.ir: inspect the known broad search response; do not submit a guessed car-search body.
divar_summary = {"platform": "Divar.ir", "ads_extracted": 0, "p0_populated": [], "p0_unavailable": [], "p1_populated": [], "p2_found": [], "needs_investigation": True, "issue": None}
divar_raw_payloads = {}
divar_rows = []
divar_df_columns = ["source_platform", "source_listing_id", "original_url", "title", "description", "price_amount_raw", "price_currency_raw", "location_raw", "category_raw", "image_urls", "vehicle_attributes_raw", "source_sort_timestamp_raw"]

try:
    divar_url = "https://api.divar.ir/v8/postlist/w/search"
    divar_response = session.get(divar_url, timeout=30)
    print("Divar HTTP status:", divar_response.status_code)
    divar_response.raise_for_status()
    divar_json = divar_response.json()

    def _divar_walk(node):
        if isinstance(node, dict):
            yield node
            for value in node.values():
                yield from _divar_walk(value)
        elif isinstance(node, list):
            for value in node:
                yield from _divar_walk(value)

    divar_nodes = list(_divar_walk(divar_json))
    divar_post_metadata = []
    for node in divar_nodes:
        if isinstance(node.get("posts_metadata"), list):
            divar_post_metadata.extend(x for x in node["posts_metadata"] if isinstance(x, dict))

    def _divar_meta_category(meta):
        category = meta.get("category")
        if isinstance(category, str):
            return category
        extra = meta.get("extra_info")
        if isinstance(extra, str):
            try:
                parsed = json.loads(extra)
                return parsed.get("category") if isinstance(parsed, dict) else None
            except (ValueError, TypeError):
                return None
        return None

    def _divar_car_category(category):
        if not isinstance(category, str):
            return False
        slug = category.strip().lower()
        known_car_slugs = {"car", "cars", "car-sell", "car-sale", "car-buy", "used-car", "used-cars", "automobile", "auto", "vehicle"}
        return slug in known_car_slugs or slug.startswith("car-") or slug.endswith("-car") or "automobile" in slug

    divar_car_metadata = [m for m in divar_post_metadata if _divar_car_category(_divar_meta_category(m))]
    divar_car_tokens = {m.get("token") for m in divar_car_metadata if isinstance(m.get("token"), str)}
    divar_candidates = []
    for node in divar_nodes:
        token = node.get("token") or node.get("post_token")
        if token in divar_car_tokens and any(k in node for k in ("web_info", "title", "name", "description", "offers", "price", "images")):
            divar_candidates.append(node)

    # Inspect any structured SEO listing records too, but do not treat these fragments
    # as complete posts unless they match a car-category token and expose the full ad object.
    for node in divar_nodes:
        if isinstance(node.get("linked_data"), list):
            for linked in node["linked_data"]:
                if isinstance(linked, dict) and _divar_car_category(linked.get("category")):
                    token = (linked.get("url") or "").rstrip("/").split("/")[-1]
                    if token in divar_car_tokens:
                        divar_candidates.append(linked)

    divar_seen = set()
    for ad in divar_candidates:
        token = ad.get("token") or ad.get("post_token")
        if token in divar_seen:
            continue
        divar_seen.add(token)
        web_info = ad.get("web_info") if isinstance(ad.get("web_info"), dict) else {}
        offers = ad.get("offers") if isinstance(ad.get("offers"), dict) else {}
        public_url = ad.get("url") if isinstance(ad.get("url"), str) and ad.get("url").startswith("https://") else None
        images = ad.get("images") or ad.get("image")
        if isinstance(images, str):
            image_urls = [images] if images.startswith("http") else None
        elif isinstance(images, list):
            image_urls = [x for x in images if isinstance(x, str) and x.startswith("http")]
        else:
            image_urls = None
        price = ad.get("price")
        if price is None:
            price = offers.get("price")
        category = ad.get("category")
        if category is None:
            category = next((_divar_meta_category(m) for m in divar_car_metadata if m.get("token") == token), None)
        row = {
            "source_platform": "Divar.ir",
            "source_listing_id": token,
            "original_url": public_url,
            "title": ad.get("title") or ad.get("name") or web_info.get("title"),
            "description": ad.get("description"),
            "price_amount_raw": price,
            "price_currency_raw": offers.get("priceCurrency"),
            "location_raw": ad.get("location") or web_info.get("city_persian"),
            "category_raw": category,
            "image_urls": image_urls,
            "vehicle_attributes_raw": json.dumps(ad.get("attributes"), ensure_ascii=False) if ad.get("attributes") is not None else None,
            "source_sort_timestamp_raw": ad.get("sort_date") or ad.get("sort"),
            "extra_info_raw": ad.get("extra_info"),
        }
        divar_rows.append(row)
        divar_raw_payloads[token] = ad
        if len(divar_rows) == 3:
            break

    if not divar_rows:
        category_values = list(dict.fromkeys(_divar_meta_category(m) for m in divar_post_metadata if _divar_meta_category(m) is not None))
        divar_jli_category = None
        for node in divar_nodes:
            jli = node.get("jli")
            if isinstance(jli, dict):
                category_obj = jli.get("category")
                divar_jli_category = category_obj.get("value") if isinstance(category_obj, dict) else category_obj
                break
        divar_summary["issue"] = "The response is a broad search feed, not a verified car-specific result; no complete car-ad payload was identified. The earlier POST had no body, and no valid car-search filter/body is established in the notebook."
        print("Selected category in response:", divar_jli_category)
        print("Categories seen in post metadata:", category_values)
        print("Car-category metadata tokens:", len(divar_car_tokens))
        print("POST follow-up skipped: the previous body-less POST did not establish a valid search payload; no request body was guessed.")
    else:
        divar_summary["issue"] = None

except Exception as exc:
    divar_summary["issue"] = f"Fetch or parse failed: {type(exc).__name__}: {exc}"
    print("Divar experiment issue:", divar_summary["issue"])

divar_df = pd.DataFrame(divar_rows, columns=divar_df_columns)
divar_p0_fields = ["source_listing_id", "original_url", "title", "description", "price_amount_raw", "price_currency_raw", "location_raw", "category_raw"]
divar_p1_fields = ["image_urls", "vehicle_attributes_raw", "source_sort_timestamp_raw"]
def _divar_nonempty(value):
    return value is not None and value != ""
divar_summary["ads_extracted"] = len(divar_rows)
divar_summary["p0_populated"] = [f for f in divar_p0_fields if any(_divar_nonempty(r.get(f)) for r in divar_rows)]
divar_summary["p0_unavailable"] = [f for f in divar_p0_fields if f not in divar_summary["p0_populated"]]
divar_summary["p1_populated"] = [f for f in divar_p1_fields if any(_divar_nonempty(r.get(f)) for r in divar_rows)]
divar_summary["p2_found"] = [f for f in ["extra_info_raw", "source_sort_timestamp_raw"] if any(_divar_nonempty(r.get(f)) for r in divar_rows)]
divar_summary["needs_investigation"] = not bool(divar_rows)
print("Divar extraction report")
print("Complete car advertisements extracted:", divar_summary["ads_extracted"])
print("P0 populated:", divar_summary["p0_populated"])
print("P0 unavailable:", divar_summary["p0_unavailable"])
print("P1 populated:", divar_summary["p1_populated"])
print("P2/source-specific fields found:", divar_summary["p2_found"])
print("Issue:", divar_summary["issue"])
if divar_rows:
    example_id = divar_rows[0].get("source_listing_id")
    print("Example complete raw advertisement JSON for", example_id, ":")
    print(json.dumps(divar_raw_payloads.get(example_id), ensure_ascii=False, indent=2))
divar_df

## Sheypoor — experimental fetch and extraction

In [ ]:
# Sheypoor: use one of the previously investigated car-search URLs and keep the sample small.
sheypoor_summary = {"platform": "Sheypoor", "ads_extracted": 0, "p0_populated": [], "p0_unavailable": [], "p1_populated": [], "p2_found": [], "needs_investigation": True, "issue": None}
sheypoor_raw_payloads = {}
sheypoor_rows = []
sheypoor_columns = ["source_platform", "source_listing_id", "original_url", "title", "description", "manufacturer_raw", "manufacturer_category_raw", "model_raw", "trim_raw", "year_raw", "mileage_raw", "price_raw", "price_amount_raw", "price_currency_raw", "location_raw", "category_raw", "fuel_raw", "transmission_raw", "body_type_raw", "body_condition_raw", "color_raw", "image_urls", "image_count", "seller_type", "seller_id", "seller_contact_raw", "seller_logo_url", "time_passed_label_raw", "sort_raw", "vehicle_attributes_raw", "source_attributes_raw", "video_count", "paid_tags_raw", "price_tag_raw", "source_extra_raw"]

try:
    # Exact investigated filter/page URL; one request only.
    sheypoor_url = "https://www.sheypoor.com/api/v10.0.0/search/iran/car?f=%5B1.0%2C%201793807126545%5D_465914613n_1793795579.366213&p=4"
    sheypoor_response = session.get(sheypoor_url, timeout=30)
    print("Sheypoor HTTP status:", sheypoor_response.status_code)
    sheypoor_response.raise_for_status()
    sheypoor_json = sheypoor_response.json()

    def _sheypoor_walk(node):
        if isinstance(node, dict):
            yield node
            for value in node.values():
                yield from _sheypoor_walk(value)
        elif isinstance(node, list):
            for value in node:
                yield from _sheypoor_walk(value)

    sheypoor_nodes = list(_sheypoor_walk(sheypoor_json))
    sheypoor_ads = []
    sheypoor_seen = set()
    for node in sheypoor_nodes:
        if (node.get("type") != "nativeAd" and isinstance(node.get("id"), (str, int))
                and isinstance(node.get("attributes"), dict)
                and isinstance(node.get("fullAttributes"), list)):
            listing_id = str(node["id"])
            if listing_id not in sheypoor_seen:
                sheypoor_ads.append(node)
                sheypoor_seen.add(listing_id)
            if len(sheypoor_ads) == 3:
                break

    def _sheypoor_nonempty(value):
        return value is not None and value != ""

    def _sheypoor_attr(ad, *keys):
        for attribute in ad.get("fullAttributes", []):
            if isinstance(attribute, dict) and attribute.get("key") in keys and _sheypoor_nonempty(attribute.get("value")):
                return attribute.get("value")
        return None

    for ad in sheypoor_ads:
        attrs = ad.get("attributes") or {}
        category_list = attrs.get("categories") if isinstance(attrs.get("categories"), list) else []
        category_raw = [{"name": c.get("name"), "id": c.get("id")} for c in category_list if isinstance(c, dict)] or None
        manufacturer_category_raw = category_list[-1].get("name") if len(category_list) >= 3 and any(isinstance(c, dict) and c.get("name") == "خودرو" for c in category_list) else None
        raw_prices = attrs.get("price")
        price_object = raw_prices[0] if isinstance(raw_prices, list) and raw_prices else None
        image_tree = attrs.get("images")
        image_urls = []
        if isinstance(image_tree, dict):
            thumbnails = image_tree.get("thumbnails")
            if isinstance(thumbnails, dict):
                image_urls.extend(v for v in thumbnails.values() if isinstance(v, str) and v.startswith("http"))
        elif isinstance(image_tree, list):
            image_urls.extend(v for v in image_tree if isinstance(v, str) and v.startswith("http"))
        image_urls = list(dict.fromkeys(image_urls)) or None
        public_url = attrs.get("url") if isinstance(attrs.get("url"), str) and attrs.get("url").startswith("https://") else None
        row = {
            "source_platform": "Sheypoor",
            "source_listing_id": str(ad.get("id")),
            "original_url": public_url,
            "title": attrs.get("title"),
            "description": ad.get("description"),
            "manufacturer_raw": manufacturer_category_raw,
            "manufacturer_category_raw": manufacturer_category_raw,
            "model_raw": _sheypoor_attr(ad, "مدل خودرو", "مدل"),
            "trim_raw": None,
            "year_raw": _sheypoor_attr(ad, "سال تولید (چهار رقمی)"),
            "mileage_raw": _sheypoor_attr(ad, "کیلومتر"),
            "price_raw": price_object if price_object is not None else raw_prices,
            "price_amount_raw": price_object.get("amount") if isinstance(price_object, dict) else None,
            "price_currency_raw": price_object.get("currency") if isinstance(price_object, dict) else None,
            "location_raw": attrs.get("location"),
            "category_raw": category_raw,
            "fuel_raw": _sheypoor_attr(ad, "نوع سوخت"),
            "transmission_raw": _sheypoor_attr(ad, "گیربکس"),
            "body_type_raw": _sheypoor_attr(ad, "نوع شاسی"),
            "body_condition_raw": _sheypoor_attr(ad, "وضعیت بدنه"),
            "color_raw": _sheypoor_attr(ad, "رنگ"),
            "image_urls": image_urls,
            "image_count": attrs.get("imageCount"),
            "seller_type": None,
            "seller_id": None,
            "seller_contact_raw": attrs.get("telephone"),
            "seller_logo_url": attrs.get("shopLogo"),
            "time_passed_label_raw": attrs.get("timePassedLabel"),
            "sort_raw": ad.get("sort"),
            "vehicle_attributes_raw": json.dumps(ad.get("fullAttributes"), ensure_ascii=False),
            "source_attributes_raw": json.dumps(attrs, ensure_ascii=False),
            "video_count": attrs.get("videoCount"),
            "paid_tags_raw": json.dumps(attrs.get("paidTags"), ensure_ascii=False) if attrs.get("paidTags") is not None else None,
            "price_tag_raw": attrs.get("priceTag"),
            "source_extra_raw": json.dumps({k: v for k, v in ad.items() if k not in {"attributes", "fullAttributes", "description"}}, ensure_ascii=False),
        }
        sheypoor_rows.append(row)
        sheypoor_raw_payloads[row["source_listing_id"]] = ad

    # Capture pagination metadata only when present in the response.
    sheypoor_pagination = {}
    for node in sheypoor_nodes:
        if isinstance(node.get("meta"), dict):
            sheypoor_pagination["meta"] = node["meta"]
        for key in ("p", "items_per_page", "total"):
            if key in node:
                sheypoor_pagination[key] = node[key]
    if not sheypoor_ads:
        sheypoor_summary["issue"] = "No records matching the captured normal-ad structure (id, attributes, fullAttributes) were found."

except Exception as exc:
    sheypoor_summary["issue"] = f"Fetch or parse failed: {type(exc).__name__}: {exc}"
    print("Sheypoor experiment issue:", sheypoor_summary["issue"])

sheypoor_df = pd.DataFrame(sheypoor_rows, columns=sheypoor_columns)
sheypoor_p0_fields = ["source_listing_id", "original_url", "title", "description", "manufacturer_raw", "model_raw", "year_raw", "mileage_raw", "price_raw", "location_raw"]
sheypoor_p1_fields = ["trim_raw", "fuel_raw", "transmission_raw", "body_type_raw", "body_condition_raw", "color_raw", "image_urls", "image_count", "seller_type", "seller_id", "seller_contact_raw", "seller_logo_url", "time_passed_label_raw"]
def _sheypoor_present(value):
    return value is not None and value != "" and value != [] and value != {}
sheypoor_summary["ads_extracted"] = len(sheypoor_rows)
sheypoor_summary["p0_populated"] = [f for f in sheypoor_p0_fields if any(_sheypoor_present(r.get(f)) for r in sheypoor_rows)]
sheypoor_summary["p0_unavailable"] = [f for f in sheypoor_p0_fields if f not in sheypoor_summary["p0_populated"]]
sheypoor_summary["p1_populated"] = [f for f in sheypoor_p1_fields if any(_sheypoor_present(r.get(f)) for r in sheypoor_rows)]
sheypoor_summary["p2_found"] = [f for f in ["vehicle_attributes_raw", "source_attributes_raw", "sort_raw", "video_count", "paid_tags_raw", "price_tag_raw", "source_extra_raw"] if any(_sheypoor_present(r.get(f)) for r in sheypoor_rows)]
sheypoor_summary["needs_investigation"] = not bool(sheypoor_rows)
print("Sheypoor extraction report")
print("Advertisements extracted:", sheypoor_summary["ads_extracted"])
print("P0 populated:", sheypoor_summary["p0_populated"])
print("P0 unavailable:", sheypoor_summary["p0_unavailable"])
print("P1 populated:", sheypoor_summary["p1_populated"])
print("P2/source-specific fields found:", sheypoor_summary["p2_found"])
print("Pagination metadata observed:", sheypoor_pagination if "sheypoor_pagination" in globals() else None)
print("Public URL verified from attributes.url:", sum(r.get("original_url") is not None for r in sheypoor_rows), "of", len(sheypoor_rows))
print("Issue:", sheypoor_summary["issue"])
if sheypoor_rows:
    example_id = sheypoor_rows[0].get("source_listing_id")
    print("Example complete raw advertisement JSON for", example_id, ":")
    print(json.dumps(sheypoor_raw_payloads.get(example_id), ensure_ascii=False, indent=2))
sheypoor_df

## Karnameh — experimental fetch and extraction

In [ ]:
# Karnameh: parse current car_posts cards; request detail JSON only if the response exposes a listing UUID.
karnameh_summary = {"platform": "Karnameh", "ads_extracted": 0, "p0_populated": [], "p0_unavailable": [], "p1_populated": [], "p2_found": [], "needs_investigation": True, "issue": None}
karnameh_raw_payloads = {}
karnameh_raw_unkeyed_records = []
karnameh_rows = []
karnameh_selected_cards = []
karnameh_detail_responses = {}
karnameh_columns = ["source_platform", "source_listing_id", "original_url", "title", "description", "brand_raw", "brand_name_fa", "brand_name_en", "model_raw", "model_name_fa", "model_name_en", "brand_model_raw", "trim_raw", "year_raw", "mileage_raw", "price_raw", "location_raw", "color_raw", "transmission_raw", "image_urls", "is_leasing_available", "has_inspection", "prepayment_amount_raw", "widget_label_raw", "state_raw", "plan_type_raw", "other_card_fields_raw"]


def _karnameh_nonempty(value):
    return value is not None and value != ""


def _karnameh_uuid_like(value):
    return isinstance(value, str) and len(value) == 36 and value.count("-") == 4


def _karnameh_get_listing_id(record):
    for key in ("id", "uuid", "post_id", "post_uuid", "car_post_id", "car_post_uuid", "guid"):
        value = record.get(key)
        if _karnameh_uuid_like(value):
            return value
    return None


def _karnameh_walk(node):
    if isinstance(node, dict):
        yield node
        for value in node.values():
            yield from _karnameh_walk(value)
    elif isinstance(node, list):
        for value in node:
            yield from _karnameh_walk(value)


def _karnameh_detail_object(payload):
    candidates = []
    for node in _karnameh_walk(payload):
        if any(key in node for key in ("brand_name_fa", "brand_name_en", "model_name_fa", "model_name_en")):
            score = sum(key in node for key in ("brand_name_fa", "brand_name_en", "model_name_fa", "model_name_en", "price", "year", "usage", "description", "car_images", "images"))
            candidates.append((score, node))
    return max(candidates, key=lambda item: item[0])[1] if candidates else {}


try:
    karnameh_list_url = "https://api-gw.karnameh.com/post-storage/car-posts/car-post-list/"
    karnameh_list_params = {
        "size": 20,
        "start": 0,
        "sort": "newest",
        "relevant": "false",
        "is_guaranteed": "false",
        "next_set": ["1793682405657", "1"],
        "user_id": "<karnameh_user_id>",
    }
    karnameh_list_response = session.get(karnameh_list_url, params=karnameh_list_params, timeout=30)
    print("Karnameh listing HTTP status:", karnameh_list_response.status_code)
    karnameh_list_response.raise_for_status()
    karnameh_list_json = karnameh_list_response.json()
    karnameh_cards = karnameh_list_json.get("car_posts", []) if isinstance(karnameh_list_json, dict) else []
    print("Current listing count metadata:", {key: karnameh_list_json.get(key) for key in ("total", "pages", "start", "size") if key in karnameh_list_json})

    if isinstance(karnameh_cards, list):
        for card in karnameh_cards:
            if isinstance(card, dict) and _karnameh_nonempty(card.get("title")) and card.get("price") is not None:
                karnameh_selected_cards.append((card, _karnameh_get_listing_id(card)))
            if len(karnameh_selected_cards) == 3:
                break

    for card, listing_id in karnameh_selected_cards:
        detail_payload = None
        detail_obj = {}
        if listing_id is not None:
            # This detail route/build ID was observed in the earlier notebook investigation.
            detail_url = f"https://karnameh.com/_next/data/nQ1KnoV5mUaQDMuVRBHOO/used-cars/{listing_id}.json"
            try:
                detail_response = session.get(detail_url, params={"id": listing_id}, timeout=30)
                print("Karnameh detail HTTP status for", listing_id, ":", detail_response.status_code)
                detail_response.raise_for_status()
                detail_payload = detail_response.json()
                karnameh_detail_responses[listing_id] = detail_payload
                detail_obj = _karnameh_detail_object(detail_payload)
            except Exception as detail_exc:
                print("Detail request issue for", listing_id, ":", f"{type(detail_exc).__name__}: {detail_exc}")

        # Prefer actual detail fields if present; otherwise use the current listing card as-is.
        combined = dict(card)
        combined.update({key: value for key, value in detail_obj.items() if value is not None})
        image_urls = []
        if isinstance(combined.get("image"), str) and combined["image"].startswith("http"):
            image_urls.append(combined["image"])
        if isinstance(combined.get("images"), list):
            image_urls.extend(value for value in combined["images"] if isinstance(value, str) and value.startswith("http"))
        if isinstance(combined.get("car_images"), list):
            for image in combined["car_images"]:
                if isinstance(image, dict):
                    image_urls.extend(value for key, value in image.items() if key in ("url", "main_url") and isinstance(value, str) and value.startswith("http"))
        image_urls = list(dict.fromkeys(image_urls)) or None

        # Only accept a public page URL explicitly supplied in the source JSON.
        original_url = None
        for key in ("original_url", "public_url", "share_url", "canonical_url", "url", "link"):
            value = combined.get(key)
            if isinstance(value, str) and value.startswith("https://karnameh.com/") and "_next/data/" not in value:
                original_url = value
                break

        row = {
            "source_platform": "Karnameh",
            "source_listing_id": listing_id,
            "original_url": original_url,
            "title": combined.get("title"),
            "description": combined.get("description"),
            "brand_raw": combined.get("brand_name_fa") if _karnameh_nonempty(combined.get("brand_name_fa")) else combined.get("brand_name_en"),
            "brand_name_fa": combined.get("brand_name_fa"),
            "brand_name_en": combined.get("brand_name_en"),
            "model_raw": combined.get("model_name_fa") if _karnameh_nonempty(combined.get("model_name_fa")) else combined.get("model_name_en"),
            "model_name_fa": combined.get("model_name_fa"),
            "model_name_en": combined.get("model_name_en"),
            "brand_model_raw": combined.get("brand_model"),
            "trim_raw": combined.get("type_name_fa") if _karnameh_nonempty(combined.get("type_name_fa")) else combined.get("type_name_en"),
            "year_raw": combined.get("year"),
            "mileage_raw": combined.get("usage"),
            "price_raw": combined.get("price"),
            "location_raw": combined.get("city_name_fa"),
            "color_raw": combined.get("color"),
            "transmission_raw": combined.get("gearbox_name_fa") if _karnameh_nonempty(combined.get("gearbox_name_fa")) else combined.get("gearbox"),
            "image_urls": image_urls,
            "is_leasing_available": combined.get("is_leasing_available"),
            "has_inspection": combined.get("has_inspection"),
            "prepayment_amount_raw": combined.get("prepayment_amount"),
            "widget_label_raw": combined.get("widget_label"),
            "state_raw": combined.get("state"),
            "plan_type_raw": combined.get("plan_type"),
            "other_card_fields_raw": json.dumps({key: value for key, value in combined.items() if key not in {"images", "car_images"}}, ensure_ascii=False),
        }
        karnameh_rows.append(row)
        if listing_id is not None:
            karnameh_raw_payloads[listing_id] = {"listing_card": card, "detail_response": detail_payload}
        else:
            # No listing ID is present in this card; retain the exact full card without fabricating a key.
            karnameh_raw_unkeyed_records.append({"listing_card": card, "detail_response": detail_payload})

except Exception as exc:
    karnameh_summary["issue"] = f"Listing fetch or parse failed: {type(exc).__name__}: {exc}"
    print("Karnameh experiment issue:", karnameh_summary["issue"])

karnameh_df = pd.DataFrame(karnameh_rows, columns=karnameh_columns)
karnameh_p0_fields = ["source_listing_id", "original_url", "title", "description", "brand_raw", "model_raw", "year_raw", "mileage_raw", "price_raw", "location_raw"]
karnameh_p1_fields = ["trim_raw", "color_raw", "transmission_raw", "image_urls", "is_leasing_available", "has_inspection"]
karnameh_summary["ads_extracted"] = len(karnameh_rows)
karnameh_summary["p0_populated"] = [field for field in karnameh_p0_fields if any(_karnameh_nonempty(row.get(field)) for row in karnameh_rows)]
karnameh_summary["p0_unavailable"] = [field for field in karnameh_p0_fields if field not in karnameh_summary["p0_populated"]]
karnameh_summary["p1_populated"] = [field for field in karnameh_p1_fields if any(_karnameh_nonempty(row.get(field)) for row in karnameh_rows)]
karnameh_summary["p2_found"] = [field for field in ["prepayment_amount_raw", "widget_label_raw", "state_raw", "plan_type_raw", "other_card_fields_raw"] if any(_karnameh_nonempty(row.get(field)) for row in karnameh_rows)]
karnameh_summary["needs_investigation"] = not bool(karnameh_rows) or any(row.get("source_listing_id") is None or row.get("original_url") is None for row in karnameh_rows)
if karnameh_summary["issue"] is None and karnameh_rows and karnameh_summary["needs_investigation"]:
    karnameh_summary["issue"] = "The listing cards contain useful car attributes but no listing UUID or public page URL; detail calls were skipped rather than misusing concierge_sale_token."
print("Karnameh extraction report")
print("Advertisements extracted from car_posts:", karnameh_summary["ads_extracted"])
print("P0 populated:", karnameh_summary["p0_populated"])
print("P0 unavailable:", karnameh_summary["p0_unavailable"])
print("P1 populated:", karnameh_summary["p1_populated"])
print("P2/source-specific fields found:", karnameh_summary["p2_found"])
print("Verified public URLs:", sum(row.get("original_url") is not None for row in karnameh_rows), "of", len(karnameh_rows))
print("Current card exposes listing UUID:", any(row.get("source_listing_id") is not None for row in karnameh_rows))
print("Issue:", karnameh_summary["issue"])
if karnameh_rows:
    print("Example complete original car_posts record:")
    print(json.dumps(karnameh_raw_unkeyed_records[0]["listing_card"] if karnameh_raw_unkeyed_records else next(iter(karnameh_raw_payloads.values()))["listing_card"], ensure_ascii=False, indent=2))
karnameh_df

## Hamrah Mechanic — experimental fetch and extraction

In [ ]:
# Hamrah Mechanic: fetch one investigated inventory page and retain up to three non-banner car records.
hamrah_summary = {"platform": "Hamrah Mechanic", "ads_extracted": 0, "p0_populated": [], "p0_unavailable": [], "p1_populated": [], "p2_found": [], "needs_investigation": True, "issue": None}
hamrah_raw_payloads = {}
hamrah_rows = []
hamrah_columns = ["source_platform", "source_listing_id", "original_url", "exhibition_detail_url_raw", "title", "description", "manufacturer_raw", "model_raw", "model_persian_raw", "trim_raw", "year_raw", "mileage_raw", "price_raw", "location_raw", "fuel_raw", "transmission_raw", "body_type_raw", "body_condition_raw", "color_raw", "seller_type", "seller_id", "image_urls", "image_count", "is_sold_raw", "leasing_option_raw", "fast_sell_raw", "in_auto_center_raw", "badge_type_raw", "footer_features_raw", "other_inventory_fields_raw"]


def _hamrah_walk(node):
    if isinstance(node, dict):
        yield node
        for value in node.values():
            yield from _hamrah_walk(value)
    elif isinstance(node, list):
        for value in node:
            yield from _hamrah_walk(value)


def _hamrah_nonempty(value):
    return value is not None and value != ""


try:
    hamrah_url = "https://www.hamrah-mechanic.com/api/v1/common/newexhibition"
    hamrah_response = session.get(hamrah_url, params={"page": 2, "size": 24}, timeout=30)
    print("Hamrah Mechanic HTTP status:", hamrah_response.status_code)
    hamrah_response.raise_for_status()
    hamrah_json = hamrah_response.json()
    hamrah_candidates = []
    hamrah_seen = set()
    for node in _hamrah_walk(hamrah_json):
        if isinstance(node.get("orderId"), (str, int)) and _hamrah_nonempty(node.get("carNamePersian")):
            if node.get("isSold") is True:
                continue
            key = str(node["orderId"])
            if key not in hamrah_seen:
                hamrah_candidates.append(node)
                hamrah_seen.add(key)
            if len(hamrah_candidates) == 3:
                break

    for ad in hamrah_candidates:
        relative_url = ad.get("exhibitionDetailUrl")
        public_url = None
        if isinstance(relative_url, str) and relative_url.startswith("/cars-for-sale/"):
            public_url = "https://www.hamrah-mechanic.com" + relative_url
        elif isinstance(relative_url, str) and relative_url.startswith("https://www.hamrah-mechanic.com/"):
            public_url = relative_url
        image_url = ad.get("imageUrl")
        row = {
            "source_platform": "Hamrah Mechanic",
            "source_listing_id": ad.get("orderId"),
            "original_url": public_url,
            "exhibition_detail_url_raw": relative_url,
            "title": ad.get("carNamePersian"),
            "description": None,
            "manufacturer_raw": ad.get("brandEnglishName"),
            "model_raw": ad.get("modelEnglishName") if _hamrah_nonempty(ad.get("modelEnglishName")) else ad.get("carNamePersian"),
            "model_persian_raw": ad.get("carNamePersian"),
            "trim_raw": ad.get("carTypeName"),
            "year_raw": ad.get("carYear"),
            "mileage_raw": ad.get("km"),
            "price_raw": ad.get("price"),
            "location_raw": ad.get("carLocation"),
            "fuel_raw": None,
            "transmission_raw": ad.get("gearBoxPersian"),
            "body_type_raw": None,
            "body_condition_raw": None,
            "color_raw": ad.get("carColorName"),
            "seller_type": None,
            "seller_id": None,
            "image_urls": [image_url] if isinstance(image_url, str) and image_url.startswith("http") else None,
            "image_count": None,
            "is_sold_raw": ad.get("isSold"),
            "leasing_option_raw": ad.get("leasingOption"),
            "fast_sell_raw": ad.get("fastSell"),
            "in_auto_center_raw": ad.get("inAutoCenter"),
            "badge_type_raw": ad.get("badgeType"),
            "footer_features_raw": json.dumps(ad.get("footerFeatures"), ensure_ascii=False) if ad.get("footerFeatures") is not None else None,
            "other_inventory_fields_raw": json.dumps({key: value for key, value in ad.items() if key not in {"imageUrl", "footerFeatures"}}, ensure_ascii=False),
        }
        hamrah_rows.append(row)
        hamrah_raw_payloads[str(ad["orderId"])] = ad

    if not hamrah_rows:
        hamrah_summary["issue"] = "No non-sold inventory objects matching orderId and carNamePersian were found in the page response."

except Exception as exc:
    hamrah_summary["issue"] = f"Fetch or parse failed: {type(exc).__name__}: {exc}"
    print("Hamrah Mechanic experiment issue:", hamrah_summary["issue"])

hamrah_df = pd.DataFrame(hamrah_rows, columns=hamrah_columns)
hamrah_p0_fields = ["source_listing_id", "original_url", "title", "description", "manufacturer_raw", "model_raw", "year_raw", "mileage_raw", "price_raw", "location_raw"]
hamrah_p1_fields = ["trim_raw", "fuel_raw", "transmission_raw", "body_type_raw", "body_condition_raw", "color_raw", "seller_type", "seller_id", "image_urls", "image_count", "is_sold_raw", "leasing_option_raw", "fast_sell_raw"]
hamrah_summary["ads_extracted"] = len(hamrah_rows)
hamrah_summary["p0_populated"] = [field for field in hamrah_p0_fields if any(_hamrah_nonempty(row.get(field)) for row in hamrah_rows)]
hamrah_summary["p0_unavailable"] = [field for field in hamrah_p0_fields if field not in hamrah_summary["p0_populated"]]
hamrah_summary["p1_populated"] = [field for field in hamrah_p1_fields if any(_hamrah_nonempty(row.get(field)) for row in hamrah_rows)]
hamrah_summary["p2_found"] = [field for field in ["in_auto_center_raw", "badge_type_raw", "footer_features_raw", "other_inventory_fields_raw"] if any(_hamrah_nonempty(row.get(field)) for row in hamrah_rows)]
hamrah_summary["needs_investigation"] = not bool(hamrah_rows) or any(row.get("original_url") is None for row in hamrah_rows)
print("Hamrah Mechanic extraction report")
print("Advertisements extracted:", hamrah_summary["ads_extracted"])
print("P0 populated:", hamrah_summary["p0_populated"])
print("P0 unavailable:", hamrah_summary["p0_unavailable"])
print("P1 populated:", hamrah_summary["p1_populated"])
print("P2/source-specific fields found:", hamrah_summary["p2_found"])
print("Public URLs verified by combining observed domain and exhibitionDetailUrl path:", sum(row.get("original_url") is not None for row in hamrah_rows), "of", len(hamrah_rows))
print("Issue:", hamrah_summary["issue"])
if hamrah_rows:
    example_id = str(hamrah_rows[0].get("source_listing_id"))
    print("Example complete raw inventory JSON for", example_id, ":")
    print(json.dumps(hamrah_raw_payloads.get(example_id), ensure_ascii=False, indent=2))
hamrah_df

## ICMI experimental coverage summary

In [ ]:
# Side-by-side coverage from the five independent extraction experiments; no new fetches.
icmi_source_summaries = [bama_summary, divar_summary, sheypoor_summary, karnameh_summary, hamrah_summary]
icmi_comparison_rows = []
for summary in icmi_source_summaries:
    icmi_comparison_rows.append({
        "source_platform": summary.get("platform"),
        "ads_extracted": summary.get("ads_extracted"),
        "P0 fields populated": ", ".join(summary.get("p0_populated", [])) or "None",
        "P0 fields unavailable": ", ".join(summary.get("p0_unavailable", [])) or "None",
        "P1 fields populated": ", ".join(summary.get("p1_populated", [])) or "None",
        "P2/source-specific fields found": ", ".join(summary.get("p2_found", [])) or "None",
        "needs further investigation": summary.get("needs_investigation"),
        "issue": summary.get("issue"),
    })
icmi_comparison_df = pd.DataFrame(icmi_comparison_rows)
icmi_comparison_df

# Major scraping attempt

## Bulk Collection Experiment — Initial Dataset

In [ ]:
import os
import time
from datetime import datetime, timezone
import email.utils
from urllib.parse import urlparse

# Earlier experiment cells may have outputs without having run in this kernel.
if "requests" not in globals():
    import requests
if "json" not in globals():
    import json
if "pd" not in globals():
    import pandas as pd
if "session" not in globals():
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "fa-IR,fa;q=0.9,en;q=0.8",
    })

# Conservative, configurable limits for this experiment.
TARGET_LISTINGS = 100
MAX_PAGES_PER_PLATFORM = 12
BAMA_PAGE_SIZE = 12          # Established endpoint behavior
SHEYPOOR_PAGE_SIZE = 24      # Observed items_per_page
HAMRAH_PAGE_SIZE = 24        # Established inventory page size
REQUEST_TIMEOUT_SECONDS = 30
REQUEST_DELAY_SECONDS = 2.0
MAX_RETRIES = 3               # Retries after the first attempt
RETRY_BACKOFF_BASE_SECONDS = 1.0
EXTRACTION_VERSION = "icmi-bulk-v1"

# Inspect the notebook's current project working directory before creating outputs.
PROJECT_ROOT = os.path.abspath(os.getcwd())
print("Project working directory:", PROJECT_ROOT)
print("Existing top-level entries:", sorted(os.listdir(PROJECT_ROOT)))

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
EXPORT_DIR = os.path.join(DATA_DIR, "exports")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
print("Output directories:", EXPORT_DIR, "and", RAW_DIR)

ICMI_OUTPUT_PATHS = {
    "Bama.ir": {
        "csv": os.path.join(EXPORT_DIR, "bama_listings.csv"),
        "jsonl": os.path.join(RAW_DIR, "bama_listings.jsonl"),
    },
    "Sheypoor": {
        "csv": os.path.join(EXPORT_DIR, "sheypoor_listings.csv"),
        "jsonl": os.path.join(RAW_DIR, "sheypoor_listings.jsonl"),
    },
    "Hamrah Mechanic": {
        "csv": os.path.join(EXPORT_DIR, "hamrah_mechanic_listings.csv"),
        "jsonl": os.path.join(RAW_DIR, "hamrah_mechanic_listings.jsonl"),
    },
}

ICMI_COMMON_COLUMNS = [
    "source_platform", "source_listing_id", "original_url", "title", "description",
    "manufacturer_raw", "brand_raw", "model_raw", "trim_raw", "year_raw",
    "mileage_raw", "price_raw", "location_raw", "fuel_raw", "transmission_raw",
    "body_type_raw", "body_condition_raw", "color_raw", "seller_type", "seller_id",
    "image_urls", "image_count", "source_published_at", "source_modified_at",
    "first_seen_at", "last_seen_at", "scraped_at", "extraction_version", "raw_payload_json",
]
ICMI_P0_FIELDS = [
    "source_listing_id", "original_url", "title", "description", "manufacturer_raw",
    "brand_raw", "model_raw", "trim_raw", "year_raw", "mileage_raw", "price_raw", "location_raw",
]


def _icmi_json_text(value):
    return None if value is None else json.dumps(value, ensure_ascii=False, default=str)


def _icmi_has_value(value):
    if value is None:
        return False
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, (list, tuple, dict, set)):
        return bool(value)
    return True


def _icmi_valid_public_url(value):
    if not isinstance(value, str):
        return False
    parsed = urlparse(value.strip())
    return parsed.scheme in ("http", "https") and bool(parsed.netloc)


def _icmi_retry_after_seconds(response):
    value = response.headers.get("Retry-After") if response is not None else None
    if not value:
        return None
    try:
        return max(0.0, float(value.strip()))
    except (TypeError, ValueError):
        try:
            retry_at = email.utils.parsedate_to_datetime(value)
            if retry_at.tzinfo is None:
                retry_at = retry_at.replace(tzinfo=timezone.utc)
            return max(0.0, (retry_at - datetime.now(timezone.utc)).total_seconds())
        except (TypeError, ValueError, OverflowError):
            return None


def _icmi_request_json(url, params, platform, page_label):
    """Sequential GET with bounded retries; returns payload, response, error, attempts, failed attempts."""
    failed_attempts = 0
    response = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = session.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
        except Exception as exc:
            failed_attempts += 1
            error = f"{type(exc).__name__}: {exc}"
            if attempt < MAX_RETRIES:
                wait = RETRY_BACKOFF_BASE_SECONDS * (2 ** attempt)
                print(f"{platform} page {page_label}: network failure; retry {attempt + 1}/{MAX_RETRIES} in {wait:.1f}s ({error})")
                time.sleep(wait)
                continue
            return None, None, error, attempt + 1, failed_attempts

        status = response.status_code
        if response.ok:
            try:
                return response.json(), response, None, attempt + 1, failed_attempts
            except Exception as exc:
                failed_attempts += 1
                return None, response, f"Invalid JSON response: {type(exc).__name__}: {exc}", attempt + 1, failed_attempts

        error = f"HTTP {status}: {response.text[:300]!r}"
        failed_attempts += 1
        retryable = status == 429 or 500 <= status <= 599
        if retryable and attempt < MAX_RETRIES:
            backoff = RETRY_BACKOFF_BASE_SECONDS * (2 ** attempt)
            retry_after = _icmi_retry_after_seconds(response)
            wait = max(backoff, retry_after or 0.0)
            print(f"{platform} page {page_label}: HTTP {status}; retry {attempt + 1}/{MAX_RETRIES} in {wait:.1f}s")
            time.sleep(wait)
            continue
        return None, response, error, attempt + 1, failed_attempts

    return None, response, "Request attempts exhausted", MAX_RETRIES + 1, failed_attempts


def _icmi_image_url_count(value):
    if not _icmi_has_value(value):
        return 0
    try:
        urls = json.loads(value) if isinstance(value, str) else value
    except (TypeError, ValueError):
        urls = [value]
    if isinstance(urls, str):
        urls = [urls]
    if isinstance(urls, dict):
        urls = list(urls.values())
    if not isinstance(urls, (list, tuple)):
        return 0
    return sum(1 for item in urls if isinstance(item, str) and _icmi_valid_public_url(item))


def _icmi_has_price(row):
    value = row.get("price_raw")
    if not _icmi_has_value(value):
        return False
    if isinstance(value, str):
        text = value.strip()
        if text.casefold() in {"توافقی", "نامشخص", "تماس بگیرید", "n/a", "none", "null"}:
            return False
        # Bama uses a zero amount for explicitly negotiable listings; retain the raw value but don't count it as a stated price.
        if text.replace(",", "").isdigit() and int(text.replace(",", "")) == 0 and str(row.get("price_type_raw", "")).casefold() == "negotiable":
            return False
    return True


def _icmi_save_platform(df, platform):
    """Write CSV/JSONL and verify both artifacts; return only verified save statuses."""
    paths = ICMI_OUTPUT_PATHS[platform]
    result = {"csv_path": paths["csv"], "jsonl_path": paths["jsonl"], "csv_saved": False, "jsonl_saved": False,
              "csv_bytes": None, "jsonl_bytes": None, "csv_rows": None, "jsonl_rows": None, "save_errors": []}
    try:
        df.to_csv(paths["csv"], index=False, encoding="utf-8-sig")
        result["csv_saved"] = os.path.isfile(paths["csv"])
        if result["csv_saved"]:
            result["csv_bytes"] = os.path.getsize(paths["csv"])
            result["csv_rows"] = len(df)
    except Exception as exc:
        result["save_errors"].append(f"CSV: {type(exc).__name__}: {exc}")

    try:
        with open(paths["jsonl"], "w", encoding="utf-8") as output_file:
            for row in df.to_dict(orient="records"):
                raw_text = row.get("raw_payload_json")
                raw_payload = json.loads(raw_text) if isinstance(raw_text, str) else None
                record = {
                    "source_platform": row.get("source_platform"),
                    "source_listing_id": row.get("source_listing_id"),
                    "raw_payload": raw_payload,
                }
                output_file.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
        result["jsonl_saved"] = os.path.isfile(paths["jsonl"])
        if result["jsonl_saved"]:
            result["jsonl_bytes"] = os.path.getsize(paths["jsonl"])
            with open(paths["jsonl"], "r", encoding="utf-8") as check_file:
                result["jsonl_rows"] = sum(1 for _ in check_file)
    except Exception as exc:
        result["save_errors"].append(f"JSONL: {type(exc).__name__}: {exc}")

    print(f"{platform} CSV: {result['csv_path']} | verified={result['csv_saved']} | bytes={result['csv_bytes']} | rows={result['csv_rows']}")
    print(f"{platform} JSONL: {result['jsonl_path']} | verified={result['jsonl_saved']} | bytes={result['jsonl_bytes']} | rows={result['jsonl_rows']}")
    if result["save_errors"]:
        print("Save errors:", result["save_errors"])
    return result


def _icmi_quality_counts(df):
    row_records = df.to_dict(orient="records")
    return {
        "rows": len(df),
        "valid_ids": sum(isinstance(row.get("source_listing_id"), str) and bool(row.get("source_listing_id", "").strip()) for row in row_records),
        "original_urls": sum(_icmi_valid_public_url(row.get("original_url")) for row in row_records),
        "descriptions": sum(_icmi_has_value(row.get("description")) for row in row_records),
        "prices": sum(_icmi_has_price(row) for row in row_records),
        "year_and_mileage": sum(_icmi_has_value(row.get("year_raw")) and _icmi_has_value(row.get("mileage_raw")) for row in row_records),
        "image_listings": sum(_icmi_image_url_count(row.get("image_urls")) > 0 for row in row_records),
        "missing_p0_percent": {
            field: round(100.0 * sum(not _icmi_has_value(row.get(field)) for row in row_records) / len(row_records), 1) if row_records else None
            for field in ICMI_P0_FIELDS
        },
    }

print("Configured target:", TARGET_LISTINGS, "per platform; page cap:", MAX_PAGES_PER_PLATFORM)
print("Request timeout:", REQUEST_TIMEOUT_SECONDS, "seconds; delay:", REQUEST_DELAY_SECONDS, "seconds; retries:", MAX_RETRIES)
print("Created output folders:", os.path.isdir(EXPORT_DIR), os.path.isdir(RAW_DIR))

### Bama\.ir — Bulk Collection

In [ ]:
# Bama.ir bulk collection: pageIndex is zero-based, pageSize=12 as established above.
bama_bulk_summary = {
    "platform": "Bama.ir", "target_listings": TARGET_LISTINGS, "pages_requested": 0,
    "pages_attempted": 0, "raw_listing_records_received": 0, "actual_unique_listings": 0,
    "duplicate_count": 0, "failed_requests": 0, "failed_pages": [], "unkeyed_listings": 0,
    "access_restricted": False, "issue": None, "target_reached": False,
}
bama_bulk_rows = []
bama_bulk_id_to_row = {}
bama_bulk_seen_pages = set()
bama_bulk_files = None


def _bama_bulk_walk(node):
    if isinstance(node, dict):
        yield node
        for child in node.values():
            yield from _bama_bulk_walk(child)
    elif isinstance(node, list):
        for child in node:
            yield from _bama_bulk_walk(child)


def _bama_bulk_collect_ads(payload):
    ads = []
    for node in _bama_bulk_walk(payload):
        detail = node.get("detail") if isinstance(node, dict) else None
        if node.get("type") == "ad" and isinstance(detail, dict) and detail.get("type") in (None, "car"):
            ads.append(node)
    return ads


bama_bulk_url = "https://bama.ir/cad/api/search"
bama_bulk_page_attempts = 0
for bama_page_index in range(MAX_PAGES_PER_PLATFORM):
    if bama_page_index in bama_bulk_seen_pages:
        print("Bama.ir: page already processed; stopping to avoid a repeated request:", bama_page_index)
        break
    if bama_bulk_page_attempts:
        time.sleep(REQUEST_DELAY_SECONDS)
    bama_bulk_seen_pages.add(bama_page_index)
    bama_bulk_page_attempts += 1
    bama_bulk_summary["pages_attempted"] += 1
    bama_payload, bama_response, bama_error, bama_attempts, bama_failed_attempts = _icmi_request_json(
        bama_bulk_url,
        params={"pageIndex": bama_page_index, "pageSize": BAMA_PAGE_SIZE},
        platform="Bama.ir",
        page_label=bama_page_index,
    )
    bama_bulk_summary["failed_requests"] += bama_failed_attempts
    if bama_error is not None:
        bama_status = bama_response.status_code if bama_response is not None else None
        bama_failure = {"page": bama_page_index, "status": bama_status, "error": bama_error, "attempts": bama_attempts}
        bama_bulk_summary["failed_pages"].append(bama_failure)
        print("Bama.ir failed page:", bama_failure)
        if bama_status in (401, 403, 407, 451):
            bama_bulk_summary["access_restricted"] = True
            bama_bulk_summary["issue"] = f"Access restriction (HTTP {bama_status}); stopped Bama requests."
            break
        if bama_status is not None and 400 <= bama_status < 500:
            bama_bulk_summary["issue"] = f"Non-retryable HTTP {bama_status}; stopped Bama requests."
            break
        continue

    bama_bulk_summary["pages_requested"] += 1
    try:
        bama_page_ads = _bama_bulk_collect_ads(bama_payload)
        bama_bulk_summary["raw_listing_records_received"] += len(bama_page_ads)
        print(f"Bama.ir pageIndex={bama_page_index}: {len(bama_page_ads)} ad records")
        if not bama_page_ads:
            print("Bama.ir returned no car advertisements; pagination complete.")
            break

        for bama_ad in bama_page_ads:
            if len(bama_bulk_id_to_row) >= TARGET_LISTINGS:
                break
            bama_detail = bama_ad.get("detail") or {}
            bama_specs = bama_ad.get("specs") if isinstance(bama_ad.get("specs"), dict) else {}
            bama_dealer = bama_ad.get("dealer")
            bama_metadata = bama_ad.get("metadata") if isinstance(bama_ad.get("metadata"), dict) else {}
            bama_price = bama_ad.get("price") if bama_ad.get("price") is not None else bama_detail.get("price")
            bama_id_value = bama_detail.get("code")
            bama_listing_id = str(bama_id_value).strip() if bama_id_value is not None and str(bama_id_value).strip() else None
            bama_identity = ("Bama.ir", bama_listing_id) if bama_listing_id is not None else None
            bama_observed_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

            if bama_identity is not None and bama_identity in bama_bulk_id_to_row:
                bama_bulk_summary["duplicate_count"] += 1
                bama_bulk_rows[bama_bulk_id_to_row[bama_identity]]["last_seen_at"] = bama_observed_at
                continue

            bama_canonical = bama_metadata.get("canonical")
            bama_relative_url = bama_detail.get("url")
            if isinstance(bama_canonical, str) and _icmi_valid_public_url(bama_canonical):
                bama_original_url = bama_canonical
            elif isinstance(bama_relative_url, str) and bama_relative_url.startswith("/"):
                # The endpoint's relative listing path and bama.ir domain were directly observed in this notebook.
                bama_original_url = "https://bama.ir" + bama_relative_url
            elif _icmi_valid_public_url(bama_relative_url):
                bama_original_url = bama_relative_url
            else:
                bama_original_url = None

            bama_image_urls = []
            if isinstance(bama_detail.get("image"), str) and _icmi_valid_public_url(bama_detail["image"]):
                bama_image_urls.append(bama_detail["image"])
            bama_images = bama_ad.get("images")
            if isinstance(bama_images, list):
                for bama_image in bama_images:
                    if isinstance(bama_image, dict):
                        bama_image_urls.extend(value for value in bama_image.values() if isinstance(value, str) and _icmi_valid_public_url(value))
                    elif isinstance(bama_image, str) and _icmi_valid_public_url(bama_image):
                        bama_image_urls.append(bama_image)
            bama_image_urls = list(dict.fromkeys(bama_image_urls))

            bama_brand_fa = bama_detail.get("brand_fa")
            bama_brand_en = bama_detail.get("brand")
            bama_row = {
                "source_platform": "Bama.ir",
                "source_listing_id": bama_listing_id,
                "original_url": bama_original_url,
                "title": bama_detail.get("title"),
                "description": bama_detail.get("description"),
                "manufacturer_raw": bama_brand_fa if _icmi_has_value(bama_brand_fa) else bama_brand_en,
                "brand_raw": bama_brand_fa if _icmi_has_value(bama_brand_fa) else bama_brand_en,
                "model_raw": bama_detail.get("subtitle"),
                "trim_raw": bama_detail.get("trim"),
                "year_raw": bama_detail.get("year"),
                "mileage_raw": bama_detail.get("mileage"),
                "price_raw": bama_price.get("price") if isinstance(bama_price, dict) else bama_price,
                "location_raw": bama_detail.get("location"),
                "fuel_raw": bama_detail.get("fuel"),
                "transmission_raw": bama_detail.get("transmission"),
                "body_type_raw": bama_detail.get("body_type_fa") if _icmi_has_value(bama_detail.get("body_type_fa")) else bama_detail.get("body_type"),
                "body_condition_raw": bama_detail.get("body_status"),
                "color_raw": bama_detail.get("color"),
                "seller_type": bama_dealer.get("type") if isinstance(bama_dealer, dict) else None,
                "seller_id": str(bama_dealer.get("id")) if isinstance(bama_dealer, dict) and bama_dealer.get("id") is not None else None,
                "image_urls": _icmi_json_text(bama_image_urls) if bama_image_urls else None,
                "image_count": bama_detail.get("image_count"),
                "source_published_at": bama_detail.get("time"),
                "source_modified_at": bama_detail.get("modified_date"),
                "first_seen_at": bama_observed_at,
                "last_seen_at": bama_observed_at,
                "scraped_at": bama_observed_at,
                "extraction_version": EXTRACTION_VERSION,
                "raw_payload_json": _icmi_json_text(bama_ad),
                "brand_en_raw": bama_brand_en,
                "price_type_raw": bama_price.get("type") if isinstance(bama_price, dict) else None,
                "price_details_json": _icmi_json_text(bama_price),
                "bama_rank": bama_detail.get("rank"),
                "bama_badge": bama_detail.get("badge"),
                "bama_pin": bama_detail.get("pin"),
                "bama_cursor": bama_ad.get("cursor"),
                "bama_generation_raw": bama_detail.get("generation"),
                "bama_body_color_raw": bama_detail.get("body_color"),
                "bama_inside_color_raw": bama_detail.get("inside_color"),
                "bama_cylinder_raw": bama_detail.get("cylinder_fa"),
                "bama_fuel_consumption_raw": bama_specs.get("fuel"),
                "bama_specs_json": _icmi_json_text(bama_specs) if bama_specs else None,
                "bama_dealer_json": _icmi_json_text(bama_dealer) if bama_dealer is not None else None,
                "bama_metadata_json": _icmi_json_text(bama_metadata) if bama_metadata else None,
                "bama_breadcrumb_json": _icmi_json_text(bama_ad.get("breadcrump")) if bama_ad.get("breadcrump") is not None else None,
                "bama_images_json": _icmi_json_text(bama_images) if bama_images is not None else None,
                "bama_source_ad_json": _icmi_json_text(bama_ad),
            }
            bama_bulk_rows.append(bama_row)
            if bama_identity is not None:
                bama_bulk_id_to_row[bama_identity] = len(bama_bulk_rows) - 1
            else:
                bama_bulk_summary["unkeyed_listings"] += 1

        bama_bulk_summary["actual_unique_listings"] = sum(
            isinstance(row.get("source_listing_id"), str) and bool(row["source_listing_id"].strip())
            for row in bama_bulk_rows
        )
        if bama_bulk_summary["actual_unique_listings"] >= TARGET_LISTINGS:
            print("Bama.ir target reached.")
            break
    except Exception as exc:
        bama_failure = {"page": bama_page_index, "status": bama_response.status_code if bama_response else None,
                        "error": f"Extraction error: {type(exc).__name__}: {exc}", "attempts": bama_attempts}
        bama_bulk_summary["failed_pages"].append(bama_failure)
        print("Bama.ir extraction failed on page:", bama_failure)

bama_extra_columns = [
    "brand_en_raw", "price_type_raw", "price_details_json", "bama_rank", "bama_badge", "bama_pin",
    "bama_cursor", "bama_generation_raw", "bama_body_color_raw", "bama_inside_color_raw",
    "bama_cylinder_raw", "bama_fuel_consumption_raw", "bama_specs_json", "bama_dealer_json",
    "bama_metadata_json", "bama_breadcrumb_json", "bama_images_json", "bama_source_ad_json",
]
bama_df = pd.DataFrame(bama_bulk_rows)
for bama_column in ICMI_COMMON_COLUMNS + bama_extra_columns:
    if bama_column not in bama_df.columns:
        bama_df[bama_column] = None
bama_df = bama_df[ICMI_COMMON_COLUMNS + bama_extra_columns]
bama_bulk_quality = _icmi_quality_counts(bama_df)
bama_bulk_summary["actual_unique_listings"] = bama_bulk_quality["valid_ids"]
bama_bulk_summary["target_reached"] = bama_bulk_summary["actual_unique_listings"] >= TARGET_LISTINGS
if bama_bulk_summary["failed_pages"] and bama_bulk_summary["issue"] is None:
    bama_bulk_summary["issue"] = "One or more pages failed; retained and saved all successfully extracted listings."
bama_bulk_files = _icmi_save_platform(bama_df, "Bama.ir")
bama_bulk_summary.update({
    "listings_with_valid_ids": bama_bulk_quality["valid_ids"],
    "listings_with_original_urls": bama_bulk_quality["original_urls"],
    "listings_with_descriptions": bama_bulk_quality["descriptions"],
    "listings_with_prices": bama_bulk_quality["prices"],
    "listings_with_year_and_mileage": bama_bulk_quality["year_and_mileage"],
    "listings_with_image_urls": bama_bulk_quality["image_listings"],
    "missing_p0_percent": bama_bulk_quality["missing_p0_percent"],
    "csv_saved": bama_bulk_files["csv_saved"], "jsonl_saved": bama_bulk_files["jsonl_saved"],
})
print("\nBama.ir collection quality report")
print("Successfully requested pages:", bama_bulk_summary["pages_requested"], "| attempted:", bama_bulk_summary["pages_attempted"])
print("Raw ad records received:", bama_bulk_summary["raw_listing_records_received"])
print("Unique advertisements with IDs:", bama_bulk_summary["actual_unique_listings"], "| unkeyed rows retained:", bama_bulk_summary["unkeyed_listings"])
print("Duplicates encountered:", bama_bulk_summary["duplicate_count"], "| failed request attempts:", bama_bulk_summary["failed_requests"])
print("Failed pages:", bama_bulk_summary["failed_pages"])
print("Listings with valid IDs / public URLs / descriptions / prices / year+ mileage / image URLs:",
      bama_bulk_summary["listings_with_valid_ids"], bama_bulk_summary["listings_with_original_urls"],
      bama_bulk_summary["listings_with_descriptions"], bama_bulk_summary["listings_with_prices"],
      bama_bulk_summary["listings_with_year_and_mileage"], bama_bulk_summary["listings_with_image_urls"])
print("P0 missing-field percentages:", bama_bulk_summary["missing_p0_percent"])
print("Target reached:", bama_bulk_summary["target_reached"], "| access restricted:", bama_bulk_summary["access_restricted"], "| issue:", bama_bulk_summary["issue"])
print("Bama sample rows (up to 3):")
print(bama_df[["source_listing_id", "title", "brand_raw", "model_raw", "year_raw", "mileage_raw", "price_raw", "location_raw", "original_url"]].iloc[:3].to_string(index=False))

### Sheypoor — Bulk Collection

In [ ]:
# Sheypoor bulk collection: begin at the previously investigated valid page (p=4),
# retain the observed f filter, and advance p sequentially.
SHEYPOOR_FILTER = "[1.0, 1793807126545]_465914613n_1793795579.366213"
SHEYPOOR_START_PAGE = 4
sheypoor_bulk_summary = {
    "platform": "Sheypoor", "target_listings": TARGET_LISTINGS, "pages_requested": 0,
    "pages_attempted": 0, "raw_listing_records_received": 0, "actual_unique_listings": 0,
    "duplicate_count": 0, "failed_requests": 0, "failed_pages": [], "unkeyed_listings": 0,
    "access_restricted": False, "issue": None, "target_reached": False,
    "pagination_metadata": {},
}
sheypoor_bulk_rows = []
sheypoor_bulk_id_to_row = {}
sheypoor_bulk_seen_pages = set()
sheypoor_bulk_files = None
sheypoor_current_filter = SHEYPOOR_FILTER


def _sheypoor_bulk_walk(node):
    if isinstance(node, dict):
        yield node
        for child in node.values():
            yield from _sheypoor_bulk_walk(child)
    elif isinstance(node, list):
        for child in node:
            yield from _sheypoor_bulk_walk(child)


def _sheypoor_bulk_attr(ad, *keys):
    for attribute in ad.get("fullAttributes", []):
        if isinstance(attribute, dict) and attribute.get("key") in keys and _icmi_has_value(attribute.get("value")):
            return attribute.get("value")
    return None


def _sheypoor_bulk_is_car(ad):
    attributes = ad.get("attributes") if isinstance(ad.get("attributes"), dict) else {}
    categories = attributes.get("categories") if isinstance(attributes.get("categories"), list) else []
    names = {str(item.get("name", "")).strip().casefold() for item in categories if isinstance(item, dict)}
    # Require the captured automobile category branch; exclude native ads and non-car search noise.
    return "خودرو" in names or "car" in names or "cars" in names


def _sheypoor_bulk_ads(payload):
    ads = []
    for node in _sheypoor_bulk_walk(payload):
        if (node.get("type") == "normal" and isinstance(node.get("id"), (str, int))
                and isinstance(node.get("attributes"), dict) and isinstance(node.get("fullAttributes"), list)
                and _sheypoor_bulk_is_car(node)):
            ads.append(node)
    return ads


sheypoor_bulk_url = "https://www.sheypoor.com/api/v10.0.0/search/iran/car"
sheypoor_bulk_page_attempts = 0
for sheypoor_page in range(SHEYPOOR_START_PAGE, SHEYPOOR_START_PAGE + MAX_PAGES_PER_PLATFORM):
    if sheypoor_page in sheypoor_bulk_seen_pages:
        print("Sheypoor: page already processed; stopping to avoid a repeated request:", sheypoor_page)
        break
    if sheypoor_bulk_page_attempts:
        time.sleep(REQUEST_DELAY_SECONDS)
    sheypoor_bulk_seen_pages.add(sheypoor_page)
    sheypoor_bulk_page_attempts += 1
    sheypoor_bulk_summary["pages_attempted"] += 1
    sheypoor_payload, sheypoor_response, sheypoor_error, sheypoor_attempts, sheypoor_failed_attempts = _icmi_request_json(
        sheypoor_bulk_url,
        params={"f": sheypoor_current_filter, "p": sheypoor_page},
        platform="Sheypoor",
        page_label=sheypoor_page,
    )
    sheypoor_bulk_summary["failed_requests"] += sheypoor_failed_attempts
    if sheypoor_error is not None:
        sheypoor_status = sheypoor_response.status_code if sheypoor_response is not None else None
        sheypoor_failure = {"page": sheypoor_page, "status": sheypoor_status, "error": sheypoor_error, "attempts": sheypoor_attempts}
        sheypoor_bulk_summary["failed_pages"].append(sheypoor_failure)
        print("Sheypoor failed page:", sheypoor_failure)
        if sheypoor_status in (401, 403, 407, 451):
            sheypoor_bulk_summary["access_restricted"] = True
            sheypoor_bulk_summary["issue"] = f"Access restriction (HTTP {sheypoor_status}); stopped Sheypoor requests."
            break
        if sheypoor_status is not None and 400 <= sheypoor_status < 500:
            sheypoor_bulk_summary["issue"] = f"Non-retryable HTTP {sheypoor_status}; stopped Sheypoor requests."
            break
        continue

    sheypoor_bulk_summary["pages_requested"] += 1
    try:
        for sheypoor_node in _sheypoor_bulk_walk(sheypoor_payload):
            if isinstance(sheypoor_node.get("meta"), dict):
                sheypoor_meta = sheypoor_node["meta"]
                sheypoor_bulk_summary["pagination_metadata"] = {
                    key: sheypoor_meta.get(key) for key in ("total", "p", "items_per_page", "api_version", "f") if key in sheypoor_meta
                }
                if _icmi_has_value(sheypoor_meta.get("f")):
                    sheypoor_current_filter = sheypoor_meta["f"]
                break
        sheypoor_page_ads = _sheypoor_bulk_ads(sheypoor_payload)
        sheypoor_bulk_summary["raw_listing_records_received"] += len(sheypoor_page_ads)
        print(f"Sheypoor p={sheypoor_page}: {len(sheypoor_page_ads)} car ad records | metadata={sheypoor_bulk_summary['pagination_metadata']}")
        if not sheypoor_page_ads:
            print("Sheypoor returned no car advertisements; pagination complete.")
            break

        sheypoor_unique_before_page = len(sheypoor_bulk_id_to_row)
        for sheypoor_ad in sheypoor_page_ads:
            if len(sheypoor_bulk_id_to_row) >= TARGET_LISTINGS:
                break
            sheypoor_attributes = sheypoor_ad.get("attributes") or {}
            sheypoor_categories = sheypoor_attributes.get("categories") if isinstance(sheypoor_attributes.get("categories"), list) else []
            sheypoor_categories_clean = [
                {key: category.get(key) for key in ("name", "id") if key in category}
                for category in sheypoor_categories if isinstance(category, dict)
            ]
            sheypoor_manufacturer = sheypoor_categories_clean[-1].get("name") if sheypoor_categories_clean else None
            sheypoor_full_attributes = sheypoor_ad.get("fullAttributes") if isinstance(sheypoor_ad.get("fullAttributes"), list) else []
            sheypoor_id_value = sheypoor_ad.get("id")
            sheypoor_listing_id = str(sheypoor_id_value).strip() if sheypoor_id_value is not None and str(sheypoor_id_value).strip() else None
            sheypoor_identity = ("Sheypoor", sheypoor_listing_id) if sheypoor_listing_id is not None else None
            sheypoor_observed_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

            if sheypoor_identity is not None and sheypoor_identity in sheypoor_bulk_id_to_row:
                sheypoor_bulk_summary["duplicate_count"] += 1
                sheypoor_bulk_rows[sheypoor_bulk_id_to_row[sheypoor_identity]]["last_seen_at"] = sheypoor_observed_at
                continue

            sheypoor_price_list = sheypoor_attributes.get("price")
            sheypoor_price_object = sheypoor_price_list[0] if isinstance(sheypoor_price_list, list) and sheypoor_price_list else None
            sheypoor_image_urls = []
            sheypoor_image_tree = sheypoor_attributes.get("images")
            if isinstance(sheypoor_image_tree, dict):
                sheypoor_thumbnails = sheypoor_image_tree.get("thumbnails")
                if isinstance(sheypoor_thumbnails, dict):
                    sheypoor_image_urls.extend(value for value in sheypoor_thumbnails.values()
                                               if isinstance(value, str) and _icmi_valid_public_url(value))
            elif isinstance(sheypoor_image_tree, list):
                sheypoor_image_urls.extend(value for value in sheypoor_image_tree
                                           if isinstance(value, str) and _icmi_valid_public_url(value))
            sheypoor_image_urls = list(dict.fromkeys(sheypoor_image_urls))
            sheypoor_url_value = sheypoor_attributes.get("url")
            sheypoor_original_url = sheypoor_url_value if _icmi_valid_public_url(sheypoor_url_value) else None
            sheypoor_price_amount = sheypoor_price_object.get("amount") if isinstance(sheypoor_price_object, dict) else None

            sheypoor_row = {
                "source_platform": "Sheypoor",
                "source_listing_id": sheypoor_listing_id,
                "original_url": sheypoor_original_url,
                "title": sheypoor_attributes.get("title"),
                "description": sheypoor_ad.get("description"),
                "manufacturer_raw": sheypoor_manufacturer,
                "brand_raw": sheypoor_manufacturer,
                "model_raw": _sheypoor_bulk_attr(sheypoor_ad, "مدل خودرو", "مدل"),
                "trim_raw": _sheypoor_bulk_attr(sheypoor_ad, "تیپ خودرو", "تیپ"),
                "year_raw": _sheypoor_bulk_attr(sheypoor_ad, "سال تولید (چهار رقمی)"),
                "mileage_raw": _sheypoor_bulk_attr(sheypoor_ad, "کیلومتر"),
                "price_raw": sheypoor_price_amount if sheypoor_price_amount is not None else (sheypoor_price_object.get("label") if isinstance(sheypoor_price_object, dict) else None),
                "location_raw": sheypoor_attributes.get("location"),
                "fuel_raw": _sheypoor_bulk_attr(sheypoor_ad, "نوع سوخت"),
                "transmission_raw": _sheypoor_bulk_attr(sheypoor_ad, "گیربکس"),
                "body_type_raw": _sheypoor_bulk_attr(sheypoor_ad, "نوع شاسی"),
                "body_condition_raw": _sheypoor_bulk_attr(sheypoor_ad, "وضعیت بدنه"),
                "color_raw": _sheypoor_bulk_attr(sheypoor_ad, "رنگ"),
                "seller_type": sheypoor_attributes.get("sellerType"),
                "seller_id": str(sheypoor_attributes.get("shopId")) if sheypoor_attributes.get("shopId") is not None else None,
                "image_urls": _icmi_json_text(sheypoor_image_urls) if sheypoor_image_urls else None,
                "image_count": sheypoor_attributes.get("imageCount"),
                "source_published_at": None,
                "source_modified_at": None,
                "first_seen_at": sheypoor_observed_at,
                "last_seen_at": sheypoor_observed_at,
                "scraped_at": sheypoor_observed_at,
                "extraction_version": EXTRACTION_VERSION,
                "raw_payload_json": _icmi_json_text(sheypoor_ad),
                "sheypoor_price_currency_raw": sheypoor_price_object.get("currency") if isinstance(sheypoor_price_object, dict) else None,
                "sheypoor_price_details_json": _icmi_json_text(sheypoor_price_list),
                "sheypoor_price_tag_raw": sheypoor_attributes.get("priceTag"),
                "sheypoor_categories_json": _icmi_json_text(sheypoor_categories_clean),
                "sheypoor_vehicle_attributes_json": _icmi_json_text(sheypoor_full_attributes),
                "sheypoor_attributes_json": _icmi_json_text(sheypoor_attributes),
                "sheypoor_sort_json": _icmi_json_text(sheypoor_ad.get("sort")) if sheypoor_ad.get("sort") is not None else None,
                "sheypoor_time_passed_label_raw": sheypoor_attributes.get("timePassedLabel"),
                "sheypoor_contact_raw": sheypoor_attributes.get("telephone"),
                "sheypoor_hide_phone_raw": sheypoor_attributes.get("hidePhoneNumber"),
                "sheypoor_shop_logo_url": sheypoor_attributes.get("shopLogo"),
                "sheypoor_video_count": sheypoor_attributes.get("videoCount"),
                "sheypoor_paid_tags_json": _icmi_json_text(sheypoor_attributes.get("paidTags")) if sheypoor_attributes.get("paidTags") is not None else None,
                "sheypoor_category_id": sheypoor_attributes.get("categoryId"),
                "sheypoor_bump_raw": sheypoor_attributes.get("bump"),
                "sheypoor_secure_purchase_raw": sheypoor_attributes.get("isSecurePurchase"),
                "sheypoor_source_ad_json": _icmi_json_text(sheypoor_ad),
            }
            sheypoor_bulk_rows.append(sheypoor_row)
            if sheypoor_identity is not None:
                sheypoor_bulk_id_to_row[sheypoor_identity] = len(sheypoor_bulk_rows) - 1
            else:
                sheypoor_bulk_summary["unkeyed_listings"] += 1

        sheypoor_bulk_summary["actual_unique_listings"] = len(sheypoor_bulk_id_to_row)
        if sheypoor_bulk_summary["actual_unique_listings"] == sheypoor_unique_before_page:
            print("Sheypoor page produced no new listing IDs; stopping to avoid repeating the same result set.")
            break
        if sheypoor_bulk_summary["actual_unique_listings"] >= TARGET_LISTINGS:
            print("Sheypoor target reached.")
            break
    except Exception as exc:
        sheypoor_failure = {"page": sheypoor_page, "status": sheypoor_response.status_code if sheypoor_response else None,
                            "error": f"Extraction error: {type(exc).__name__}: {exc}", "attempts": sheypoor_attempts}
        sheypoor_bulk_summary["failed_pages"].append(sheypoor_failure)
        print("Sheypoor extraction failed on page:", sheypoor_failure)

sheypoor_extra_columns = [
    "sheypoor_price_currency_raw", "sheypoor_price_details_json", "sheypoor_price_tag_raw",
    "sheypoor_categories_json", "sheypoor_vehicle_attributes_json", "sheypoor_attributes_json",
    "sheypoor_sort_json", "sheypoor_time_passed_label_raw", "sheypoor_contact_raw",
    "sheypoor_hide_phone_raw", "sheypoor_shop_logo_url", "sheypoor_video_count",
    "sheypoor_paid_tags_json", "sheypoor_category_id", "sheypoor_bump_raw",
    "sheypoor_secure_purchase_raw", "sheypoor_source_ad_json",
]
sheypoor_df = pd.DataFrame(sheypoor_bulk_rows)
for sheypoor_column in ICMI_COMMON_COLUMNS + sheypoor_extra_columns:
    if sheypoor_column not in sheypoor_df.columns:
        sheypoor_df[sheypoor_column] = None
sheypoor_df = sheypoor_df[ICMI_COMMON_COLUMNS + sheypoor_extra_columns]
sheypoor_bulk_quality = _icmi_quality_counts(sheypoor_df)
sheypoor_bulk_summary["actual_unique_listings"] = sheypoor_bulk_quality["valid_ids"]
sheypoor_bulk_summary["target_reached"] = sheypoor_bulk_summary["actual_unique_listings"] >= TARGET_LISTINGS
if sheypoor_bulk_summary["failed_pages"] and sheypoor_bulk_summary["issue"] is None:
    sheypoor_bulk_summary["issue"] = "One or more pages failed; retained and saved all successfully extracted listings."
sheypoor_bulk_files = _icmi_save_platform(sheypoor_df, "Sheypoor")
sheypoor_bulk_summary.update({
    "listings_with_valid_ids": sheypoor_bulk_quality["valid_ids"],
    "listings_with_original_urls": sheypoor_bulk_quality["original_urls"],
    "listings_with_descriptions": sheypoor_bulk_quality["descriptions"],
    "listings_with_prices": sheypoor_bulk_quality["prices"],
    "listings_with_year_and_mileage": sheypoor_bulk_quality["year_and_mileage"],
    "listings_with_image_urls": sheypoor_bulk_quality["image_listings"],
    "missing_p0_percent": sheypoor_bulk_quality["missing_p0_percent"],
    "csv_saved": sheypoor_bulk_files["csv_saved"], "jsonl_saved": sheypoor_bulk_files["jsonl_saved"],
})
print("\nSheypoor collection quality report")
print("Start p:", SHEYPOOR_START_PAGE, "| filter preserved:", SHEYPOOR_FILTER)
print("Successfully requested pages:", sheypoor_bulk_summary["pages_requested"], "| attempted:", sheypoor_bulk_summary["pages_attempted"])
print("Latest pagination metadata:", sheypoor_bulk_summary["pagination_metadata"])
print("Raw car-ad records received:", sheypoor_bulk_summary["raw_listing_records_received"])
print("Unique advertisements with IDs:", sheypoor_bulk_summary["actual_unique_listings"], "| unkeyed rows retained:", sheypoor_bulk_summary["unkeyed_listings"])
print("Duplicates encountered:", sheypoor_bulk_summary["duplicate_count"], "| failed request attempts:", sheypoor_bulk_summary["failed_requests"])
print("Failed pages:", sheypoor_bulk_summary["failed_pages"])
print("Listings with valid IDs / public URLs / descriptions / prices / year+ mileage / image URLs:",
      sheypoor_bulk_summary["listings_with_valid_ids"], sheypoor_bulk_summary["listings_with_original_urls"],
      sheypoor_bulk_summary["listings_with_descriptions"], sheypoor_bulk_summary["listings_with_prices"],
      sheypoor_bulk_summary["listings_with_year_and_mileage"], sheypoor_bulk_summary["listings_with_image_urls"])
print("P0 missing-field percentages:", sheypoor_bulk_summary["missing_p0_percent"])
print("Target reached:", sheypoor_bulk_summary["target_reached"], "| access restricted:", sheypoor_bulk_summary["access_restricted"], "| issue:", sheypoor_bulk_summary["issue"])
print("Sheypoor sample rows (up to 3):")
print(sheypoor_df[["source_listing_id", "title", "manufacturer_raw", "model_raw", "year_raw", "mileage_raw", "price_raw", "location_raw", "original_url"]].iloc[:3].to_string(index=False))

### Hamrah Mechanic — Bulk Collection

In [ ]:
# Hamrah Mechanic bulk collection: page 2 is the first page previously fetched in this notebook.
HAMRAH_START_PAGE = 2
hamrah_bulk_summary = {
    "platform": "Hamrah Mechanic", "target_listings": TARGET_LISTINGS, "pages_requested": 0,
    "pages_attempted": 0, "raw_listing_records_received": 0, "actual_unique_listings": 0,
    "duplicate_count": 0, "failed_requests": 0, "failed_pages": [], "unkeyed_listings": 0,
    "sold_records_skipped": 0, "access_restricted": False, "issue": None, "target_reached": False,
}
hamrah_bulk_rows = []
hamrah_bulk_id_to_row = {}
hamrah_bulk_seen_pages = set()
hamrah_bulk_files = None


def _hamrah_bulk_walk(node):
    if isinstance(node, dict):
        yield node
        for child in node.values():
            yield from _hamrah_bulk_walk(child)
    elif isinstance(node, list):
        for child in node:
            yield from _hamrah_bulk_walk(child)


def _hamrah_bulk_inventory(payload):
    vehicles = []
    for node in _hamrah_bulk_walk(payload):
        if isinstance(node.get("orderId"), (str, int)) and _icmi_has_value(node.get("carNamePersian")):
            vehicles.append(node)
    return vehicles


hamrah_bulk_url = "https://www.hamrah-mechanic.com/api/v1/common/newexhibition"
hamrah_bulk_page_attempts = 0
for hamrah_page in range(HAMRAH_START_PAGE, HAMRAH_START_PAGE + MAX_PAGES_PER_PLATFORM):
    if hamrah_page in hamrah_bulk_seen_pages:
        print("Hamrah Mechanic: page already processed; stopping to avoid a repeated request:", hamrah_page)
        break
    if hamrah_bulk_page_attempts:
        time.sleep(REQUEST_DELAY_SECONDS)
    hamrah_bulk_seen_pages.add(hamrah_page)
    hamrah_bulk_page_attempts += 1
    hamrah_bulk_summary["pages_attempted"] += 1
    hamrah_payload, hamrah_response, hamrah_error, hamrah_attempts, hamrah_failed_attempts = _icmi_request_json(
        hamrah_bulk_url,
        params={"page": hamrah_page, "size": HAMRAH_PAGE_SIZE},
        platform="Hamrah Mechanic",
        page_label=hamrah_page,
    )
    hamrah_bulk_summary["failed_requests"] += hamrah_failed_attempts
    if hamrah_error is not None:
        hamrah_status = hamrah_response.status_code if hamrah_response is not None else None
        hamrah_failure = {"page": hamrah_page, "status": hamrah_status, "error": hamrah_error, "attempts": hamrah_attempts}
        hamrah_bulk_summary["failed_pages"].append(hamrah_failure)
        print("Hamrah Mechanic failed page:", hamrah_failure)
        if hamrah_status in (401, 403, 407, 451):
            hamrah_bulk_summary["access_restricted"] = True
            hamrah_bulk_summary["issue"] = f"Access restriction (HTTP {hamrah_status}); stopped Hamrah Mechanic requests."
            break
        if hamrah_status is not None and 400 <= hamrah_status < 500:
            hamrah_bulk_summary["issue"] = f"Non-retryable HTTP {hamrah_status}; stopped Hamrah Mechanic requests."
            break
        continue

    hamrah_bulk_summary["pages_requested"] += 1
    try:
        hamrah_page_vehicles = _hamrah_bulk_inventory(hamrah_payload)
        hamrah_bulk_summary["raw_listing_records_received"] += len(hamrah_page_vehicles)
        print(f"Hamrah Mechanic page={hamrah_page}: {len(hamrah_page_vehicles)} vehicle inventory records")
        if not hamrah_page_vehicles:
            print("Hamrah Mechanic returned no vehicle inventory records; pagination complete.")
            break

        hamrah_unique_before_page = len(hamrah_bulk_id_to_row)
        for hamrah_ad in hamrah_page_vehicles:
            if len(hamrah_bulk_id_to_row) >= TARGET_LISTINGS:
                break
            if hamrah_ad.get("isSold") is True:
                hamrah_bulk_summary["sold_records_skipped"] += 1
                continue

            hamrah_id_value = hamrah_ad.get("orderId")
            hamrah_listing_id = str(hamrah_id_value).strip() if hamrah_id_value is not None and str(hamrah_id_value).strip() else None
            hamrah_identity = ("Hamrah Mechanic", hamrah_listing_id) if hamrah_listing_id is not None else None
            hamrah_observed_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
            if hamrah_identity is not None and hamrah_identity in hamrah_bulk_id_to_row:
                hamrah_bulk_summary["duplicate_count"] += 1
                hamrah_bulk_rows[hamrah_bulk_id_to_row[hamrah_identity]]["last_seen_at"] = hamrah_observed_at
                continue

            hamrah_relative_url = hamrah_ad.get("exhibitionDetailUrl")
            if isinstance(hamrah_relative_url, str) and hamrah_relative_url.startswith("/cars-for-sale/"):
                # Relative listing paths and this host were directly observed in the notebook's API response.
                hamrah_original_url = "https://www.hamrah-mechanic.com" + hamrah_relative_url
            elif _icmi_valid_public_url(hamrah_relative_url):
                hamrah_original_url = hamrah_relative_url
            else:
                hamrah_original_url = None
            hamrah_image = hamrah_ad.get("imageUrl")
            hamrah_image_urls = [hamrah_image] if isinstance(hamrah_image, str) and _icmi_valid_public_url(hamrah_image) else []
            hamrah_brand = hamrah_ad.get("brandEnglishName")
            hamrah_row = {
                "source_platform": "Hamrah Mechanic",
                "source_listing_id": hamrah_listing_id,
                "original_url": hamrah_original_url,
                "title": hamrah_ad.get("carNamePersian"),
                "description": None,
                "manufacturer_raw": hamrah_brand,
                "brand_raw": hamrah_brand,
                "model_raw": hamrah_ad.get("modelEnglishName") if _icmi_has_value(hamrah_ad.get("modelEnglishName")) else hamrah_ad.get("carNamePersian"),
                "trim_raw": hamrah_ad.get("carTypeName"),
                "year_raw": hamrah_ad.get("carYear"),
                "mileage_raw": hamrah_ad.get("km"),
                "price_raw": hamrah_ad.get("price"),
                "location_raw": hamrah_ad.get("carLocation"),
                "fuel_raw": None,
                "transmission_raw": hamrah_ad.get("gearBoxPersian"),
                "body_type_raw": None,
                "body_condition_raw": None,
                "color_raw": hamrah_ad.get("carColorName"),
                "seller_type": None,
                "seller_id": None,
                "image_urls": _icmi_json_text(hamrah_image_urls) if hamrah_image_urls else None,
                "image_count": None,
                "source_published_at": None,
                "source_modified_at": None,
                "first_seen_at": hamrah_observed_at,
                "last_seen_at": hamrah_observed_at,
                "scraped_at": hamrah_observed_at,
                "extraction_version": EXTRACTION_VERSION,
                "raw_payload_json": _icmi_json_text(hamrah_ad),
                "hamrah_car_id": hamrah_ad.get("carId"),
                "hamrah_car_type_id": hamrah_ad.get("carTypeId"),
                "hamrah_gearbox_raw": hamrah_ad.get("gearBoxPersian"),
                "hamrah_car_location_raw": hamrah_ad.get("carLocation"),
                "hamrah_leasing_option_raw": hamrah_ad.get("leasingOption"),
                "hamrah_leasing_section_tag_raw": hamrah_ad.get("leasingSectionTagEnum"),
                "hamrah_fast_sell_raw": hamrah_ad.get("fastSell"),
                "hamrah_is_sold_raw": hamrah_ad.get("isSold"),
                "hamrah_in_auto_center_raw": hamrah_ad.get("inAutoCenter"),
                "hamrah_badge_type_raw": hamrah_ad.get("badgeType"),
                "hamrah_offer_price_raw": hamrah_ad.get("offerPrice"),
                "hamrah_offer_price_difference_raw": hamrah_ad.get("offerPriceDifferencePercentage"),
                "hamrah_special_offer_end_time_raw": hamrah_ad.get("specialOfferEndTime"),
                "hamrah_coming_soon_raw": hamrah_ad.get("comingSoon"),
                "hamrah_available_colors_json": _icmi_json_text(hamrah_ad.get("availableColors")) if hamrah_ad.get("availableColors") is not None else None,
                "hamrah_footer_features_json": _icmi_json_text(hamrah_ad.get("footerFeatures")) if hamrah_ad.get("footerFeatures") is not None else None,
                "hamrah_source_inventory_json": _icmi_json_text(hamrah_ad),
            }
            hamrah_bulk_rows.append(hamrah_row)
            if hamrah_identity is not None:
                hamrah_bulk_id_to_row[hamrah_identity] = len(hamrah_bulk_rows) - 1
            else:
                hamrah_bulk_summary["unkeyed_listings"] += 1

        hamrah_bulk_summary["actual_unique_listings"] = len(hamrah_bulk_id_to_row)
        if hamrah_bulk_summary["actual_unique_listings"] == hamrah_unique_before_page:
            print("Hamrah Mechanic page produced no new listing IDs; stopping to avoid repeating the same inventory.")
            break
        if hamrah_bulk_summary["actual_unique_listings"] >= TARGET_LISTINGS:
            print("Hamrah Mechanic target reached.")
            break
    except Exception as exc:
        hamrah_failure = {"page": hamrah_page, "status": hamrah_response.status_code if hamrah_response else None,
                          "error": f"Extraction error: {type(exc).__name__}: {exc}", "attempts": hamrah_attempts}
        hamrah_bulk_summary["failed_pages"].append(hamrah_failure)
        print("Hamrah Mechanic extraction failed on page:", hamrah_failure)

hamrah_extra_columns = [
    "hamrah_car_id", "hamrah_car_type_id", "hamrah_gearbox_raw", "hamrah_car_location_raw",
    "hamrah_leasing_option_raw", "hamrah_leasing_section_tag_raw", "hamrah_fast_sell_raw",
    "hamrah_is_sold_raw", "hamrah_in_auto_center_raw", "hamrah_badge_type_raw",
    "hamrah_offer_price_raw", "hamrah_offer_price_difference_raw", "hamrah_special_offer_end_time_raw",
    "hamrah_coming_soon_raw", "hamrah_available_colors_json", "hamrah_footer_features_json",
    "hamrah_source_inventory_json",
]
hamrah_df = pd.DataFrame(hamrah_bulk_rows)
for hamrah_column in ICMI_COMMON_COLUMNS + hamrah_extra_columns:
    if hamrah_column not in hamrah_df.columns:
        hamrah_df[hamrah_column] = None
hamrah_df = hamrah_df[ICMI_COMMON_COLUMNS + hamrah_extra_columns]
hamrah_bulk_quality = _icmi_quality_counts(hamrah_df)
hamrah_bulk_summary["actual_unique_listings"] = hamrah_bulk_quality["valid_ids"]
hamrah_bulk_summary["target_reached"] = hamrah_bulk_summary["actual_unique_listings"] >= TARGET_LISTINGS
if hamrah_bulk_summary["failed_pages"] and hamrah_bulk_summary["issue"] is None:
    hamrah_bulk_summary["issue"] = "One or more pages failed; retained and saved all successfully extracted listings."
hamrah_bulk_files = _icmi_save_platform(hamrah_df, "Hamrah Mechanic")
hamrah_bulk_summary.update({
    "listings_with_valid_ids": hamrah_bulk_quality["valid_ids"],
    "listings_with_original_urls": hamrah_bulk_quality["original_urls"],
    "listings_with_descriptions": hamrah_bulk_quality["descriptions"],
    "listings_with_prices": hamrah_bulk_quality["prices"],
    "listings_with_year_and_mileage": hamrah_bulk_quality["year_and_mileage"],
    "listings_with_image_urls": hamrah_bulk_quality["image_listings"],
    "missing_p0_percent": hamrah_bulk_quality["missing_p0_percent"],
    "csv_saved": hamrah_bulk_files["csv_saved"], "jsonl_saved": hamrah_bulk_files["jsonl_saved"],
})
print("\nHamrah Mechanic collection quality report")
print("Start page:", HAMRAH_START_PAGE, "| size:", HAMRAH_PAGE_SIZE)
print("Successfully requested pages:", hamrah_bulk_summary["pages_requested"], "| attempted:", hamrah_bulk_summary["pages_attempted"])
print("Raw vehicle records received:", hamrah_bulk_summary["raw_listing_records_received"], "| sold records skipped:", hamrah_bulk_summary["sold_records_skipped"])
print("Unique advertisements with IDs:", hamrah_bulk_summary["actual_unique_listings"], "| unkeyed rows retained:", hamrah_bulk_summary["unkeyed_listings"])
print("Duplicates encountered:", hamrah_bulk_summary["duplicate_count"], "| failed request attempts:", hamrah_bulk_summary["failed_requests"])
print("Failed pages:", hamrah_bulk_summary["failed_pages"])
print("Listings with valid IDs / public URLs / descriptions / prices / year+ mileage / image URLs:",
      hamrah_bulk_summary["listings_with_valid_ids"], hamrah_bulk_summary["listings_with_original_urls"],
      hamrah_bulk_summary["listings_with_descriptions"], hamrah_bulk_summary["listings_with_prices"],
      hamrah_bulk_summary["listings_with_year_and_mileage"], hamrah_bulk_summary["listings_with_image_urls"])
print("P0 missing-field percentages:", hamrah_bulk_summary["missing_p0_percent"])
print("Target reached:", hamrah_bulk_summary["target_reached"], "| access restricted:", hamrah_bulk_summary["access_restricted"], "| issue:", hamrah_bulk_summary["issue"])
print("Hamrah Mechanic sample rows (up to 3):")
print(hamrah_df[["source_listing_id", "title", "brand_raw", "model_raw", "year_raw", "mileage_raw", "price_raw", "location_raw", "original_url"]].iloc[:3].to_string(index=False))

### Collection Summary and Data Quality Report

In [ ]:
# Final comparison, filesystem verification, and compact row previews.
icmi_platform_results = [
    ("Bama.ir", bama_df, bama_bulk_summary, bama_bulk_files),
    ("Sheypoor", sheypoor_df, sheypoor_bulk_summary, sheypoor_bulk_files),
    ("Hamrah Mechanic", hamrah_df, hamrah_bulk_summary, hamrah_bulk_files),
]
icmi_collection_rows = []


def _icmi_json_valid(value):
    if not isinstance(value, str):
        return False
    try:
        json.loads(value)
        return True
    except (TypeError, ValueError):
        return False


icmi_preview_columns = [
    "source_listing_id", "title", "manufacturer_raw", "brand_raw", "model_raw", "trim_raw",
    "year_raw", "mileage_raw", "price_raw", "location_raw", "fuel_raw", "transmission_raw", "original_url",
]

for icmi_platform, icmi_df, icmi_summary, icmi_files in icmi_platform_results:
    icmi_paths = ICMI_OUTPUT_PATHS[icmi_platform]
    icmi_csv_rows = None
    icmi_jsonl_rows = None
    icmi_csv_bytes = None
    icmi_jsonl_bytes = None
    icmi_csv_saved_verified = False
    icmi_jsonl_saved_verified = False
    icmi_file_errors = []

    if os.path.isfile(icmi_paths["csv"]):
        try:
            icmi_csv_bytes = os.path.getsize(icmi_paths["csv"])
            icmi_csv_rows = len(pd.read_csv(icmi_paths["csv"], encoding="utf-8-sig"))
            icmi_csv_saved_verified = icmi_csv_rows == len(icmi_df)
        except Exception as exc:
            icmi_file_errors.append(f"CSV verification: {type(exc).__name__}: {exc}")
    if os.path.isfile(icmi_paths["jsonl"]):
        try:
            icmi_jsonl_bytes = os.path.getsize(icmi_paths["jsonl"])
            with open(icmi_paths["jsonl"], "r", encoding="utf-8") as icmi_jsonl_file:
                icmi_jsonl_rows = sum(1 for _ in icmi_jsonl_file)
            icmi_jsonl_saved_verified = icmi_jsonl_rows == len(icmi_df)
        except Exception as exc:
            icmi_file_errors.append(f"JSONL verification: {type(exc).__name__}: {exc}")

    icmi_files["csv_saved"] = icmi_csv_saved_verified
    icmi_files["jsonl_saved"] = icmi_jsonl_saved_verified
    icmi_files["csv_bytes"] = icmi_csv_bytes
    icmi_files["jsonl_bytes"] = icmi_jsonl_bytes
    icmi_files["csv_rows"] = icmi_csv_rows
    icmi_files["jsonl_rows"] = icmi_jsonl_rows
    icmi_summary["csv_saved"] = icmi_csv_saved_verified
    icmi_summary["jsonl_saved"] = icmi_jsonl_saved_verified

    icmi_id_count = int(icmi_df["source_listing_id"].map(lambda value: isinstance(value, str) and bool(value.strip())).sum())
    icmi_duplicate_keys = int(icmi_df.dropna(subset=["source_listing_id"]).duplicated(subset=["source_platform", "source_listing_id"]).sum())
    icmi_raw_json_valid = int(icmi_df["raw_payload_json"].map(
        lambda value: isinstance(value, str) and _icmi_json_valid(value)
    ).sum()) if "_icmi_json_valid" in globals() else None

    print(f"\n{icmi_platform} verified artifacts and row-level checks")
    print("CSV:", icmi_paths["csv"], "| exists/row-count verified:", icmi_csv_saved_verified,
          "| bytes:", icmi_csv_bytes, "| rows:", icmi_csv_rows)
    print("JSONL:", icmi_paths["jsonl"], "| exists/row-count verified:", icmi_jsonl_saved_verified,
          "| bytes:", icmi_jsonl_bytes, "| rows:", icmi_jsonl_rows)
    print("IDs are strings:", icmi_id_count, "of", len(icmi_df), "| duplicate composite keys remaining:", icmi_duplicate_keys)
    if icmi_file_errors:
        print("File verification errors:", icmi_file_errors)
    print(f"{icmi_platform} first five normalized rows (selected common fields):")
    print(icmi_df.reindex(columns=icmi_preview_columns).iloc[:5].to_string(index=False))

    icmi_collection_rows.append({
        "source_platform": icmi_platform,
        "target_listings": icmi_summary.get("target_listings", TARGET_LISTINGS),
        "actual_unique_listings": icmi_summary.get("actual_unique_listings", 0),
        "pages_requested": icmi_summary.get("pages_requested", 0),
        "duplicate_count": icmi_summary.get("duplicate_count", 0),
        "failed_requests": icmi_summary.get("failed_requests", 0),
        "listings_with_descriptions": icmi_summary.get("listings_with_descriptions", 0),
        "listings_with_prices": icmi_summary.get("listings_with_prices", 0),
        "listings_with_images": icmi_summary.get("listings_with_image_urls", 0),
        "listings_with_original_urls": icmi_summary.get("listings_with_original_urls", 0),
        "csv_saved": icmi_csv_saved_verified,
        "jsonl_saved": icmi_jsonl_saved_verified,
        "target_reached": icmi_summary.get("target_reached", False),
        "p0_missing_percent": icmi_summary.get("missing_p0_percent", {}),
        "unique_id_count_in_dataframe": icmi_id_count,
        "duplicate_composite_keys_remaining": icmi_duplicate_keys,
        "csv_row_count_verified": icmi_csv_rows,
        "jsonl_row_count_verified": icmi_jsonl_rows,
        "access_restricted": icmi_summary.get("access_restricted", False),
        "issue": icmi_summary.get("issue"),
    })

icmi_collection_summary_df = pd.DataFrame(icmi_collection_rows)
print("\nCombined collection summary")
icmi_collection_summary_df

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=3376e6b2-94ef-4c60-bdb6-76fb7f1a2963' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>